# 03 — Node2Vec Ingredient Embeddings

## Goal
Learn ingredient embeddings from the frozen ingredient co-occurrence graph.
Embeddings should reflect **functional/contextual role similarity** (useful for substitution),
not just raw co-occurrence frequency.

## Frozen inputs (read-only)
- data/processed/ingredient_cooccurrence_edges.csv
  - columns: ingredient_a, ingredient_b, weight
- data/processed/recipe_ingredients.csv (not used directly in this notebook)

## Iteration 0 (baseline hyperparameters)
We start from literature-aligned defaults and a mild DFS bias to encourage role similarity.

- dimension (d): 128
- walk length (l): 60
- walks per node (r): 20
- window size (k): 10
- return parameter (p): 1
- in-out parameter (q): 2
- random seed: 42

## Evidence we will record each iteration
For a fixed probe list of ingredients, we will record:
- Top-10 nearest neighbours (cosine similarity)
- Short notes: what improved / what got worse / next change

We will only change **one parameter at a time** between iterations.


In [8]:
# -----------------------------------------------------------
# A.0 — Load the frozen ingredient co-occurrence edge list
# -----------------------------------------------------------

import pandas as pd  # pandas lets us load and inspect tabular data (CSV files)

# Path to the frozen edge list produced earlier (treat this file as read-only)
EDGES_PATH = "../data/processed/ingredient_cooccurrence_edges.csv"

# Load the CSV into a DataFrame (a table-like structure: rows + columns)
edges_df = pd.read_csv(EDGES_PATH)

# Quick sanity checks / inspection (we want to confirm the schema is as expected)
print("Loaded edges from:", EDGES_PATH)                 # show which file we loaded
print("Number of edges (rows):", len(edges_df))         # how many co-occurrence pairs
print("Columns:", list(edges_df.columns))               # confirm column names

# Show the first few rows so we can visually verify the format
edges_df.head()


Loaded edges from: ../data/processed/ingredient_cooccurrence_edges.csv
Number of edges (rows): 419913
Columns: ['ingredient_a', 'ingredient_b', 'weight']


,ingredient_a,ingredient_b,weight
0,egg,flour,27623
1,egg,milk,16380
2,flour,milk,16133
3,flour,powder,15622
4,egg,powder,13475


In [ ]:
# -----------------------------------------------------------
# A.1 — Build the weighted undirected ingredient graph
# -----------------------------------------------------------

import networkx as nx  # NetworkX gives us a graph object we can run Node2Vec on

# Create an empty undirected graph (because ingredient co-occurrence is unordered)
G = nx.Graph()

# Add all edges in one go, storing the co-occurrence count as the edge attribute "weight"
# Each row becomes an edge: ingredient_a -- ingredient_b with weight=<count>
G.add_weighted_edges_from(
    edges_df[["ingredient_a", "ingredient_b", "weight"]].itertuples(index=False, name=None),
    weight="weight"
)

# Quick visibility checks (not correctness checks)
print("Graph built.")
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

# Example: show the weight of the strongest edge we saw in the CSV preview
print("Weight(egg, flour):", G["egg"]["flour"]["weight"])


Graph built.
Number of nodes: 2698
Number of edges: 419913
Weight(egg, flour): 27623


In [10]:
# -----------------------------------------------------------
# B.0 — Train Node2Vec (Iteration 0 baseline)
# -----------------------------------------------------------

from node2vec import Node2Vec  # Node2Vec generates biased random walks + trains embeddings

# 1) Prepare random walks + training setup
node2vec = Node2Vec(
    G,                      # our ingredient graph
    dimensions=128,         # d: size of each embedding vector
    walk_length=60,         # l: steps per walk
    num_walks=20,           # r: walks started from each node
    p=1,                    # return bias (baseline)
    q=2,                    # in-out bias (mild DFS → role similarity)
    weight_key="weight",    # use edge weights when choosing next step
    workers=4,              # how many CPU cores to use
    seed=42                 # reproducibility
)

# 2) Train skip-gram (Word2Vec) on the walks to learn embeddings
model = node2vec.fit(
    window=10,              # k: context window size
    min_count=1,            # keep all ingredients (even rare ones)
    batch_words=4           # training batch size
)

print("Iteration 0 training complete.")


Computing transition probabilities: 100%|██████████| 2698/2698 [37:22<00:00,  1.20it/s]    
/Users/adarpalaoglu/Documents/cs310-safeplate/venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Generating walks (CPU: 4): 100%|██████████| 5/5 [00:37<00:00,  7.45s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception i

Iteration 0 training complete.


In [11]:
# -----------------------------------------------------------
# B.1 — Save Iteration 0 embeddings (so we don’t retrain)
# -----------------------------------------------------------

from pathlib import Path

OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save in Word2Vec text format (easy to load later)
emb_path = OUT_DIR / "ingredient_node2vec_iter0.txt"
model.wv.save_word2vec_format(str(emb_path))

print("Saved embeddings to:", emb_path)
print("Number of embeddings saved:", len(model.wv))
print("Embedding dimension:", model.vector_size)
# -----------------------------------------------------------
# B.1 — Save Iteration 0 embeddings (so we don’t retrain)
# -----------------------------------------------------------

from pathlib import Path

OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save in Word2Vec text format (easy to load later)
emb_path = OUT_DIR / "ingredient_node2vec_iter0.txt"
model.wv.save_word2vec_format(str(emb_path))

print("Saved embeddings to:", emb_path)
print("Number of embeddings saved:", len(model.wv))
print("Embedding dimension:", model.vector_size)


Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter0.txt
Number of embeddings saved: 2698
Embedding dimension: 128
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter0.txt
Number of embeddings saved: 2698
Embedding dimension: 128


In [12]:
# -----------------------------------------------------------
# C.0 — Iteration 0 evidence: nearest-neighbour sanity checks
# -----------------------------------------------------------

# A fixed probe list (keep this constant across all iterations)
# We choose ingredients that are common, substitutable, and cover different roles.
probe_ingredients = [
    "butter", "olive oil", "vegetable oil",
    "milk", "yogurt", "cream",
    "flour", "bread flour",
    "sugar", "brown sugar", "honey",
    "egg",
    "garlic", "onion",
    "chicken", "beef",
    "rice", "pasta"
]

# Some ingredients may not exist *exactly* with these spellings in your vocabulary.
# This helper checks which probes are in-vocab vs missing.
vocab = set(model.wv.key_to_index.keys())
present = [x for x in probe_ingredients if x in vocab]
missing = [x for x in probe_ingredients if x not in vocab]

print("Probe ingredients present:", present)
print("Probe ingredients missing:", missing)

# Print top-10 neighbours for each present probe
TOPN = 10
for ing in present:
    print("\n" + "=" * 60)
    print(f"Probe: {ing}")
    neighbours = model.wv.most_similar(ing, topn=TOPN)
    for rank, (nbr, sim) in enumerate(neighbours, start=1):
        print(f"{rank:02d}. {nbr:<25}  cosine={sim:.3f}")


Probe ingredients present: ['milk', 'yogurt', 'cream', 'flour', 'bread flour', 'honey', 'egg', 'garlic', 'chicken', 'beef', 'rice', 'pasta']
Probe ingredients missing: ['butter', 'olive oil', 'vegetable oil', 'sugar', 'brown sugar', 'onion']

Probe: milk
01. egg                        cosine=0.770
02. english walnut             cosine=0.758
03. coconut pecan frosting     cosine=0.749
04. flour                      cosine=0.749
05. peppermint patty           cosine=0.741
06. milky way bar              cosine=0.738
07. margarine                  cosine=0.737
08. special k cereal           cosine=0.737
09. cream                      cosine=0.721
10. real vanilla               cosine=0.714

Probe: yogurt
01. soy yogurt                 cosine=0.624
02. sumaq                      cosine=0.609
03. vegetarian sausage         cosine=0.606
04. mango pulp                 cosine=0.604
05. lamb rib chop              cosine=0.599
06. lamb shoulder              cosine=0.596
07. lemon juice           

In [13]:
# -----------------------------------------------------------
# D.0 — Iteration 1: increase DFS bias to reduce hub dominance
# (Only change: q from 2 → 4)
# -----------------------------------------------------------

from node2vec import Node2Vec
from pathlib import Path

# --- Iteration 1 hyperparameters (ONLY q changes) ---
ITER = 1
DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
WINDOW = 10
P = 1
Q = 4          # <-- the only intentional change vs Iteration 0
SEED = 42

# 1) Create Node2Vec object (walk generation setup)
node2vec_iter1 = Node2Vec(
    G,                      # ingredient graph
    dimensions=DIM,         # embedding size
    walk_length=WALK_LEN,   # steps per walk
    num_walks=NUM_WALKS,    # walks per node
    p=P,                    # return parameter (unchanged)
    q=Q,                    # in-out parameter (changed)
    weight_key="weight",    # use co-occurrence weights in transitions
    workers=2,              # more stable than 4 on Mac laptops
    seed=SEED               # reproducibility
)

# 2) Train skip-gram model on walks (learn embeddings)
# Setting workers=1 here reduces gensim thread/shutdown noise.
model_iter1 = node2vec_iter1.fit(
    window=WINDOW,          # context window
    min_count=1,            # keep all ingredients
    batch_words=4,          # stable small batches
    workers=1               # stability: avoids noisy "Exception ignored" spam
)

print(f"Iteration {ITER} training complete (p={P}, q={Q}).")

# 3) Save embeddings to disk (new filename so we can compare later)
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter1 = OUT_DIR / f"ingredient_node2vec_iter{ITER}_p{P}_q{Q}.txt"
model_iter1.wv.save_word2vec_format(str(emb_path_iter1))

print("Saved embeddings to:", emb_path_iter1)
print("Number of embeddings saved:", len(model_iter1.wv))
print("Embedding dimension:", model_iter1.vector_size)


Generating walks (CPU: 2): 100%|██████████| 10/10 [01:18<00:00,  7.89s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.ou

Iteration 1 training complete (p=1, q=4).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter1_p1_q4.txt
Number of embeddings saved: 2698
Embedding dimension: 128


In [14]:
# -----------------------------------------------------------
# D.1 — Iteration 1 evidence: nearest neighbours (same probes)
# -----------------------------------------------------------

# Reuse the same probe list (do NOT change it)
vocab1 = set(model_iter1.wv.key_to_index.keys())
present1 = [x for x in probe_ingredients if x in vocab1]
missing1 = [x for x in probe_ingredients if x not in vocab1]

print("Probe ingredients present:", present1)
print("Probe ingredients missing:", missing1)

TOPN = 10
for ing in present1:
    print("\n" + "=" * 60)
    print(f"Probe: {ing}")
    neighbours = model_iter1.wv.most_similar(ing, topn=TOPN)
    for rank, (nbr, sim) in enumerate(neighbours, start=1):
        print(f"{rank:02d}. {nbr:<25}  cosine={sim:.3f}")


Probe ingredients present: ['milk', 'yogurt', 'cream', 'flour', 'bread flour', 'honey', 'egg', 'garlic', 'chicken', 'beef', 'rice', 'pasta']
Probe ingredients missing: ['butter', 'olive oil', 'vegetable oil', 'sugar', 'brown sugar', 'onion']

Probe: milk
01. egg                        cosine=0.775
02. margarine                  cosine=0.771
03. honey graham cracker       cosine=0.768
04. egg bread                  cosine=0.758
05. puree                      cosine=0.752
06. oreo cookie pie crust      cosine=0.751
07. flour                      cosine=0.748
08. milky way bar              cosine=0.740
09. skor candy bar             cosine=0.737
10. chai tea                   cosine=0.735

Probe: yogurt
01. pomegranate syrup          cosine=0.633
02. smooth cashew butter       cosine=0.624
03. mint sauce                 cosine=0.623
04. lemon juice                cosine=0.620
05. red plum                   cosine=0.616
06. 1 fat buttermilk           cosine=0.597
07. bourbon whiskey       

In [16]:
# -----------------------------------------------------------
# E.1 — Canonical probe list (mapped to actual vocab tokens)
# -----------------------------------------------------------

probe_ingredients_canonical = [
    # fats / oils
    "real butter",
    "pure olive oil",
    "canola oil",              # neutral oil proxy for "vegetable oil"
    
    # dairy
    "milk", "yogurt", "cream",
    
    # baking staples
    "flour", "bread flour",
    "cane sugar",
    "light-brown sugar",
    "honey",
    
    # aromatics
    "garlic",
    "red onion",
    
    # proteins
    "chicken", "beef",
    
    # carbs
    "rice", "pasta"
]

def print_neighbours(model, probe_list, topn=10, title=""):
    vocab = set(model.wv.key_to_index.keys())
    present = [x for x in probe_list if x in vocab]
    missing = [x for x in probe_list if x not in vocab]
    
    if title:
        print("\n" + "#" * 70)
        print(title)
        print("#" * 70)
        
    print("Present probes:", present)
    print("Missing probes:", missing)
    
    for ing in present:
        print("\n" + "=" * 60)
        print(f"Probe: {ing}")
        for i, (nbr, sim) in enumerate(model.wv.most_similar(ing, topn=topn), start=1):
            print(f"{i:02d}. {nbr:<30} cosine={sim:.3f}")

# Run for Iteration 0 model
print_neighbours(model, probe_ingredients_canonical, topn=10, title="Iteration 0 neighbours (canonical probes)")

# Run for Iteration 1 model
print_neighbours(model_iter1, probe_ingredients_canonical, topn=10, title="Iteration 1 neighbours (canonical probes)")



######################################################################
Iteration 0 neighbours (canonical probes)
######################################################################
Present probes: ['real butter', 'pure olive oil', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: []

Probe: real butter
01. 18 table cream                 cosine=0.630
02. club cracker                   cosine=0.601
03. candy cane                     cosine=0.593
04. ready-made pie crust           cosine=0.585
05. tap water                      cosine=0.570
06. vanilla chip                   cosine=0.562
07. chocolate icing                cosine=0.559
08. cherry jam                     cosine=0.558
09. buttermilk pancake             cosine=0.556
10. egg bread                      cosine=0.555

Probe: pure olive oil
01. hazelnut oil                   cosine=0.686
02. kos

In [17]:
# -----------------------------------------------------------
# F.0 — Iteration 2: reduce context mixing by shrinking window
# (Only change: window from 10 → 5, keep p=1, q=2)
# -----------------------------------------------------------

from node2vec import Node2Vec
from pathlib import Path

# --- Iteration 2 hyperparameters (ONLY window changes vs Iteration 0) ---
ITER = 2
DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
P = 1
Q = 2               # back to the baseline exploration setting
WINDOW = 5          # <-- the only intentional change vs Iteration 0
SEED = 42

# 1) Create Node2Vec object (random walk generation setup)
node2vec_iter2 = Node2Vec(
    G,                      # ingredient graph
    dimensions=DIM,         # size of each embedding vector
    walk_length=WALK_LEN,   # steps per walk
    num_walks=NUM_WALKS,    # walks started from each node
    p=P,                    # return parameter
    q=Q,                    # in-out parameter
    weight_key="weight",    # use co-occurrence counts in transitions
    workers=2,              # stable on Mac laptops
    seed=SEED               # reproducibility
)

# 2) Train skip-gram model on the walks (learn embeddings)
# workers=1 here reduces gensim shutdown/thread noise
model_iter2 = node2vec_iter2.fit(
    window=WINDOW,          # <-- only change vs Iteration 0
    min_count=1,            # keep all ingredients
    batch_words=4,          # small stable batches
    workers=1               # stability
)

print(f"Iteration {ITER} training complete (p={P}, q={Q}, window={WINDOW}).")

# 3) Save embeddings to disk (new filename for comparison)
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter2 = OUT_DIR / f"ingredient_node2vec_iter{ITER}_p{P}_q{Q}_w{WINDOW}.txt"
model_iter2.wv.save_word2vec_format(str(emb_path_iter2))

print("Saved embeddings to:", emb_path_iter2)
print("Number of embeddings saved:", len(model_iter2.wv))
print("Embedding dimension:", model_iter2.vector_size)

# -----------------------------------------------------------
# F.1 — Iteration 2 evidence: nearest neighbours (canonical probes)
# -----------------------------------------------------------

print_neighbours(
    model_iter2,
    probe_ingredients_canonical,
    topn=10,
    title="Iteration 2 neighbours (canonical probes) — window reduced to 5"
)


Generating walks (CPU: 2): 100%|██████████| 10/10 [01:23<00:00,  8.35s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.ou

Iteration 2 training complete (p=1, q=2, window=5).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter2_p1_q2_w5.txt
Number of embeddings saved: 2698
Embedding dimension: 128

######################################################################
Iteration 2 neighbours (canonical probes) — window reduced to 5
######################################################################
Present probes: ['real butter', 'pure olive oil', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: []

Probe: real butter
01. red raspberry preserve         cosine=0.664
02. milk chocolate kisse           cosine=0.646
03. pastry crust                   cosine=0.645
04. phyllo cup                     cosine=0.637
05. milk chocolate candy bar       cosine=0.630
06. ande mint candy                cosine=0.625
07. chocolate cookie               cosine=0.623
0

In [18]:
# -----------------------------------------------------------
# G.0 — Iteration 3: remove edge weights (UNWEIGHTED walks)
# (Only change: do NOT use weight_key)
# -----------------------------------------------------------

from node2vec import Node2Vec
from pathlib import Path

# --- Iteration 3 hyperparameters (IDENTICAL to Iteration 2) ---
ITER = 3
DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
P = 1
Q = 2
WINDOW = 5
SEED = 42

# 1) Create Node2Vec object (UNWEIGHTED random walks)
node2vec_iter3 = Node2Vec(
    G,                      # ingredient graph
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    # IMPORTANT: weight_key intentionally omitted
    workers=2,
    seed=SEED
)

# 2) Train skip-gram model
model_iter3 = node2vec_iter3.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=1
)

print(f"Iteration {ITER} training complete (UNWEIGHTED walks).")

# 3) Save embeddings
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter3 = OUT_DIR / f"ingredient_node2vec_iter{ITER}_UNWEIGHTED_p{P}_q{Q}_w{WINDOW}.txt"
model_iter3.wv.save_word2vec_format(str(emb_path_iter3))

print("Saved embeddings to:", emb_path_iter3)
print("Number of embeddings saved:", len(model_iter3.wv))
print("Embedding dimension:", model_iter3.vector_size)

# -----------------------------------------------------------
# G.1 — Iteration 3 evidence: nearest neighbours (canonical probes)
# -----------------------------------------------------------

print_neighbours(
    model_iter3,
    probe_ingredients_canonical,
    topn=10,
    title="Iteration 3 neighbours (canonical probes) — UNWEIGHTED walks"
)

Generating walks (CPU: 2): 100%|██████████| 10/10 [01:25<00:00,  8.57s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.ou

Iteration 3 training complete (UNWEIGHTED walks).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter3_UNWEIGHTED_p1_q2_w5.txt
Number of embeddings saved: 2698
Embedding dimension: 128

######################################################################
Iteration 3 neighbours (canonical probes) — UNWEIGHTED walks
######################################################################
Present probes: ['real butter', 'pure olive oil', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: []

Probe: real butter
01. preserve                       cosine=0.658
02. non-hydrogenated margarine     cosine=0.641
03. dutch process cocoa            cosine=0.637
04. real vanilla extract           cosine=0.637
05. mint chocolate                 cosine=0.635
06. jumbo egg                      cosine=0.635
07. extra-large egg yolk           cosine=0

In [19]:
# -----------------------------------------------------------
# H.0 — Iteration 4: UNWEIGHTED + stronger DFS bias to push structural equivalence
# (Only change vs Iteration 3: q from 2 → 4)
# -----------------------------------------------------------

from node2vec import Node2Vec
from pathlib import Path

# --- Iteration 4 hyperparameters (ONLY q changes vs Iteration 3) ---
ITER = 4
DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
P = 1
Q = 4            # <-- ONLY intentional change vs Iteration 3 (q=2 → q=4)
WINDOW = 5
SEED = 42

# 1) Create Node2Vec object (UNWEIGHTED random walks)
# IMPORTANT: weight_key intentionally omitted to keep walks unweighted.
node2vec_iter4 = Node2Vec(
    G,                      # ingredient graph
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    workers=2,              # stable on Mac
    seed=SEED
)

# 2) Train skip-gram model
model_iter4 = node2vec_iter4.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=1               # stability: reduces shutdown/thread noise
)

print(f"Iteration {ITER} training complete (UNWEIGHTED, p={P}, q={Q}, window={WINDOW}).")

# 3) Save embeddings
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter4 = OUT_DIR / f"ingredient_node2vec_iter{ITER}_UNWEIGHTED_p{P}_q{Q}_w{WINDOW}.txt"
model_iter4.wv.save_word2vec_format(str(emb_path_iter4))

print("Saved embeddings to:", emb_path_iter4)
print("Number of embeddings saved:", len(model_iter4.wv))
print("Embedding dimension:", model_iter4.vector_size)

# -----------------------------------------------------------
# H.1 — Iteration 4 evidence: nearest neighbours (canonical probes)
# -----------------------------------------------------------

print_neighbours(
    model_iter4,
    probe_ingredients_canonical,
    topn=10,
    title="Iteration 4 neighbours (canonical probes) — UNWEIGHTED, q increased to 4"
)


Generating walks (CPU: 2): 100%|██████████| 10/10 [01:23<00:00,  8.40s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.ou

Iteration 4 training complete (UNWEIGHTED, p=1, q=4, window=5).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter4_UNWEIGHTED_p1_q4_w5.txt
Number of embeddings saved: 2698
Embedding dimension: 128

######################################################################
Iteration 4 neighbours (canonical probes) — UNWEIGHTED, q increased to 4
######################################################################
Present probes: ['real butter', 'pure olive oil', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: []

Probe: real butter
01. popcorn                        cosine=0.709
02. buttermilk pancake             cosine=0.706
03. dipping chocolate              cosine=0.694
04. milky way bar                  cosine=0.694
05. 1 fat buttermilk               cosine=0.689
06. mint chocolate                 cosine=0.686
07. crisco oil   

In [20]:
# -----------------------------------------------------------
# I.0 — Iteration 5: Association-Strength WEIGHTED graph (same params as Iteration 4)
# (Only change vs Iteration 4: switch graph source to assoc-strength weighted edges)
# -----------------------------------------------------------

import pandas as pd
import networkx as nx
from node2vec import Node2Vec
from pathlib import Path

# Path to the NEW normalised edge list (association strength weights)
EDGES_ASSOC_PATH = "../data/processed/ingredient_cooccurrence_edges_assoc_strength.csv"

# --- Iteration 5 hyperparameters (IDENTICAL to Iteration 4) ---
ITER = 5
DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
P = 1
Q = 4
WINDOW = 5
SEED = 42

# 1) Load the association-strength edge list (same columns, different meaning of "weight")
edges_assoc_df = pd.read_csv(EDGES_ASSOC_PATH)

print("Loaded association-strength edges from:", EDGES_ASSOC_PATH)
print("Number of edges (rows):", len(edges_assoc_df))
print("Columns:", list(edges_assoc_df.columns))
display(edges_assoc_df.head())

# 2) Build a NEW graph object so we do NOT overwrite the original G
# This keeps all previous iterations comparable (each one uses its own graph choice).
G_assoc = nx.Graph()

# Add edges with association-strength weights
# Each row becomes: ingredient_a -- ingredient_b with weight=<association_strength>
G_assoc.add_weighted_edges_from(
    edges_assoc_df[["ingredient_a", "ingredient_b", "weight"]].itertuples(index=False, name=None),
    weight="weight"
)

print("Association-strength graph built.")
print("Number of nodes:", G_assoc.number_of_nodes())
print("Number of edges:", G_assoc.number_of_edges())

# Optional quick check: show a familiar edge weight if it exists in this graph
# (Not all classic pairs will necessarily appear if your edge list was filtered earlier.)
if G_assoc.has_edge("egg", "flour"):
    print("AssocWeight(egg, flour):", G_assoc["egg"]["flour"]["weight"])
else:
    print("Edge (egg, flour) not found in assoc graph (this can happen if the edge list was filtered).")

# 3) Train Node2Vec on the association-strength WEIGHTED graph
# IMPORTANT: we DO use weight_key here, because the whole point is to test this weighting scheme.
node2vec_iter5 = Node2Vec(
    G_assoc,
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    weight_key="weight",   # <-- use association-strength weights during transitions
    workers=2,
    seed=SEED
)

# 4) Fit skip-gram (Word2Vec) model to learn embeddings
model_iter5 = node2vec_iter5.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=1              # stability on Mac
)

print(f"Iteration {ITER} training complete (ASSOC-WEIGHTED, p={P}, q={Q}, window={WINDOW}).")

# 5) Save embeddings (new filename to keep comparisons clean)
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter5 = OUT_DIR / f"ingredient_node2vec_iter{ITER}_ASSOC_WEIGHTED_p{P}_q{Q}_w{WINDOW}.txt"
model_iter5.wv.save_word2vec_format(str(emb_path_iter5))

print("Saved embeddings to:", emb_path_iter5)
print("Number of embeddings saved:", len(model_iter5.wv))
print("Embedding dimension:", model_iter5.vector_size)

# -----------------------------------------------------------
# I.1 — Iteration 5 evidence: nearest neighbours (canonical probes)
# -----------------------------------------------------------

print_neighbours(
    model_iter5,
    probe_ingredients_canonical,
    topn=10,
    title="Iteration 5 neighbours (canonical probes) — ASSOC-WEIGHTED graph, same params as Iteration 4"
)


Loaded association-strength edges from: ../data/processed/ingredient_cooccurrence_edges_assoc_strength.csv
Number of edges (rows): 419913
Columns: ['ingredient_a', 'ingredient_b', 'weight']


,ingredient_a,ingredient_b,weight
0,vegan cream cheese,vegan parmesan cheese,0.010417
1,corn husk,masa harina flour,0.009804
2,capicola,mortadella,0.008791
3,corn chex,wheat chex,0.008222
4,rice chex,wheat chex,0.007477


Association-strength graph built.
Number of nodes: 2698
Number of edges: 419913
AssocWeight(egg, flour): 1.0892050059415712e-05


Generating walks (CPU: 2): 100%|██████████| 10/10 [00:39<00:00,  4.00s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Iteration 5 training complete (ASSOC-WEIGHTED, p=1, q=4, window=5).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter5_ASSOC_WEIGHTED_p1_q4_w5.txt
Number of embeddings saved: 2698
Embedding dimension: 128

######################################################################
Iteration 5 neighbours (canonical probes) — ASSOC-WEIGHTED graph, same params as Iteration 4
######################################################################
Present probes: ['real butter', 'pure olive oil', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: []

Probe: real butter
01. real vanilla extract           cosine=0.454
02. mountain dew soda              cosine=0.453
03. butter recipe cake             cosine=0.429
04. fig preserve                   cosine=0.401
05. unbaked pie crust              cosine=0.395
06. real vanilla                   co

In [21]:
# -----------------------------------------------------------
# J.0 — Diagnose Iteration 5 + choose the next best move
# -----------------------------------------------------------
# Why Iteration 5 looks worse:
# Association-strength weights can make the graph extremely "spiky":
# - rare–rare pairs get disproportionately high weights
# - random walks over-weight niche edges and under-sample common functional neighbours
# Result: neighbours become quirky / noisy (e.g., "real butter" ~ "mountain dew soda")

import numpy as np
import pandas as pd

def summarize_weight_distribution(edges_df, name="edges"):
    w = edges_df["weight"].astype(float).values
    print(f"\n=== {name}: weight distribution summary ===")
    print("count:", len(w))
    for p in [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 99.5, 99.9, 100]:
        print(f"p{p:>5}: {np.percentile(w, p):.12g}")
    print("min:", w.min(), "max:", w.max())
    print("mean:", w.mean(), "median:", np.median(w))

# Compare raw-count weights vs association-strength weights
summarize_weight_distribution(edges_df, name="RAW co-occurrence (count weights)")
summarize_weight_distribution(edges_assoc_df, name="ASSOC strength weights")

# Show the top assoc-strength edges: these are often rare/rare pairs with small counts
# which can dominate walks if used directly as transition weights.
print("\nTop 15 assoc-strength edges (often rare/rare pairs):")
display(
    edges_assoc_df.sort_values("weight", ascending=False)
                 .head(15)
)

# -----------------------------------------------------------
# J.1 — Recommended next step (best outcome): keep assoc-strength for PRUNING,
# but use log-count for WALK transitions
# -----------------------------------------------------------
# This is exactly the strategy you validated earlier:
# - assoc-strength chooses which edges survive (reduces hub dominance)
# - log1p(count) provides stable transition weights (reduces rare-edge domination)

# Inputs you already have:
# - edges_assoc_df: assoc-strength weights for ALL edges
# - edges_df: raw counts (weight column == count) for ALL edges
# We will:
# 1) merge counts into assoc table
# 2) for each node, keep top-K by assoc-strength
# 3) collapse to undirected set
# 4) set final walk weights = log1p(count)
# 5) train Node2Vec on the resulting pruned+logweighted graph

TOPK = 50  # keep top 50 strongest assoc edges per node (good default)

# 1) Merge raw counts into assoc edges
raw_counts = edges_df.rename(columns={"weight": "count"})
assoc = edges_assoc_df.merge(raw_counts, on=["ingredient_a", "ingredient_b"], how="left")
print("Missing counts after merge (should be 0):", assoc["count"].isna().sum())

# 2) Build directed top-K (each node keeps its best neighbours by assoc strength)
# Make a directed view by duplicating each undirected edge both directions.
a_to_b = assoc[["ingredient_a", "ingredient_b", "weight", "count"]].copy()
b_to_a = assoc.rename(columns={"ingredient_a": "ingredient_b", "ingredient_b": "ingredient_a"})[
    ["ingredient_a", "ingredient_b", "weight", "count"]
].copy()

directed = pd.concat([a_to_b, b_to_a], ignore_index=True)

# For each source node, keep top-K outgoing edges by assoc-strength weight
directed_topk = (
    directed.sort_values(["ingredient_a", "weight"], ascending=[True, False])
            .groupby("ingredient_a", as_index=False)
            .head(TOPK)
)

print("Directed edges after top-K pruning:", len(directed_topk))

# 3) Collapse to undirected edge set (canonical ordering)
def canon_pair(row):
    a, b = row["ingredient_a"], row["ingredient_b"]
    return (a, b) if a < b else (b, a)

directed_topk["a_canon"], directed_topk["b_canon"] = zip(*directed_topk.apply(canon_pair, axis=1))

undirected_pruned = (
    directed_topk.groupby(["a_canon", "b_canon"], as_index=False)
                 .agg({"count": "max"})   # if edge appears twice, counts are same; max is safe
                 .rename(columns={"a_canon": "ingredient_a", "b_canon": "ingredient_b"})
)

print("Undirected edges after pruning:", len(undirected_pruned))

# 4) Define walk weights as log1p(count)
undirected_pruned["weight"] = np.log1p(undirected_pruned["count"].astype(float))

# Quick sanity: top weights should be common pairs, not weird rare ones
print("\nTop 10 edges after pruning (by log-count walk weight):")
display(
    undirected_pruned.sort_values("weight", ascending=False)
                    .head(10)[["ingredient_a", "ingredient_b", "weight", "count"]]
)

# 5) Save pruned edge list (so it becomes a clean frozen input for embeddings)
PRUNED_EDGES_PATH = "../data/processed/ingredient_edges_pruned_by_assoc_top50_logcount.csv"
undirected_pruned[["ingredient_a", "ingredient_b", "weight"]].to_csv(PRUNED_EDGES_PATH, index=False)
print("Saved pruned edge list to:", PRUNED_EDGES_PATH)
print("Rows saved:", len(undirected_pruned))

# -----------------------------------------------------------
# J.2 — Iteration 6: Node2Vec on PRUNED graph (assoc-pruned + logcount-walk)
# -----------------------------------------------------------

import networkx as nx
from node2vec import Node2Vec
from pathlib import Path

# --- Iteration 6 hyperparameters ---
ITER = 6
DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
P = 1
Q = 4          # keep the stronger DFS bias that helped before
WINDOW = 5
SEED = 42

# Build graph from the new pruned edge list
G_pruned = nx.Graph()
G_pruned.add_weighted_edges_from(
    undirected_pruned[["ingredient_a", "ingredient_b", "weight"]].itertuples(index=False, name=None),
    weight="weight"
)

print("\nPruned graph built.")
print("Nodes:", G_pruned.number_of_nodes())
print("Edges:", G_pruned.number_of_edges())

# Train Node2Vec using the pruned + log-count weights
node2vec_iter6 = Node2Vec(
    G_pruned,
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    weight_key="weight",   # use log-count weights for transitions
    workers=2,
    seed=SEED
)

model_iter6 = node2vec_iter6.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=1
)

print(f"Iteration {ITER} training complete (ASSOC-PRUNED + LOGCOUNT-WEIGHTED, p={P}, q={Q}, window={WINDOW}).")

# Save embeddings
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter6 = OUT_DIR / f"ingredient_node2vec_iter{ITER}_ASSOC_PRUNED_top{TOPK}_LOGCOUNT_p{P}_q{Q}_w{WINDOW}.txt"
model_iter6.wv.save_word2vec_format(str(emb_path_iter6))

print("Saved embeddings to:", emb_path_iter6)
print("Number of embeddings saved:", len(model_iter6.wv))
print("Embedding dimension:", model_iter6.vector_size)

# Evidence: neighbours for canonical probes
print_neighbours(
    model_iter6,
    probe_ingredients_canonical,
    topn=10,
    title=f"Iteration {ITER} neighbours — ASSOC-PRUNED(top{TOPK}) + LOGCOUNT walk weights"
)



=== RAW co-occurrence (count weights): weight distribution summary ===
count: 419913
p    0: 1
p    1: 1
p    5: 1
p   10: 1
p   25: 1
p   50: 2
p   75: 5
p   90: 16
p   95: 36
p   99: 188
p 99.5: 348
p 99.9: 1184.176
p  100: 27623
min: 1.0 max: 27623.0
mean: 12.90589241104705 median: 2.0

=== ASSOC strength weights: weight distribution summary ===
count: 419913
p    0: 6.1058288263e-09
p    1: 3.66700800355e-07
p    5: 1.04835611668e-06
p   10: 1.76038982151e-06
p   25: 4.00775902147e-06
p   50: 9.9025588212e-06
p   75: 2.83687943262e-05
p   90: 8.89110179658e-05
p   95: 0.000180310133429
p   99: 0.000651958484251
p 99.5: 0.000965250965251
p 99.9: 0.0018286659577
p  100: 0.0104166666667
min: 6.1058288263005205e-09 max: 0.0104166666666666
mean: 4.4785437698621914e-05 median: 9.9025588211994e-06

Top 15 assoc-strength edges (often rare/rare pairs):


,ingredient_a,ingredient_b,weight
0,vegan cream cheese,vegan parmesan cheese,0.010417
1,corn husk,masa harina flour,0.009804
2,capicola,mortadella,0.008791
3,corn chex,wheat chex,0.008222
4,rice chex,wheat chex,0.007477
5,channa dal,mung dal,0.007143
6,butter pecan cake,coconut pecan frosting,0.006803
7,paper cup,wooden popsicle stick,0.006579
8,nori,sushi rice,0.006531
9,hing,mung dal,0.006522


Missing counts after merge (should be 0): 0
Directed edges after top-K pruning: 134648
Undirected edges after pruning: 81362

Top 10 edges after pruning (by log-count walk weight):


,ingredient_a,ingredient_b,weight,count
47612,flour,powder,9.656499,15622
43828,egg,powder,9.508666,13475
47623,flour,soda,9.436519,12537
43839,egg,soda,9.311994,11069
74150,powder,soda,8.818038,6754
32698,cinnamon,nutmeg,8.513787,4982
79192,soda,vanilla,8.398184,4438
23252,carrot,celery,8.222016,3721
9674,basil,oregano,8.143517,3440
29175,chili powder,cumin,8.080856,3231


Saved pruned edge list to: ../data/processed/ingredient_edges_pruned_by_assoc_top50_logcount.csv
Rows saved: 81362

Pruned graph built.
Nodes: 2698
Edges: 81362


Generating walks (CPU: 2): 100%|██████████| 10/10 [00:05<00:00,  1.80it/s]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Iteration 6 training complete (ASSOC-PRUNED + LOGCOUNT-WEIGHTED, p=1, q=4, window=5).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter6_ASSOC_PRUNED_top50_LOGCOUNT_p1_q4_w5.txt
Number of embeddings saved: 2698
Embedding dimension: 128

######################################################################
Iteration 6 neighbours — ASSOC-PRUNED(top50) + LOGCOUNT walk weights
######################################################################
Present probes: ['real butter', 'pure olive oil', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: []

Probe: real butter
01. butter recipe cake             cosine=0.413
02. cream cheese frosting          cosine=0.407
03. chervil                        cosine=0.406
04. real maple syrup               cosine=0.399
05. frosting                       cosine=0.397
06. cream of potato soup      

In [22]:
# ===========================================================
# NEW: Node2Vec on the INGREDIENT–RECIPE BIPARTITE GRAPH
# (2-step ingredient-only walks: ingredient -> recipe -> ingredient)
# ===========================================================

import pandas as pd
import numpy as np
import networkx as nx
from node2vec import Node2Vec
from pathlib import Path

# -----------------------------------------------------------
# A.0 — Load the frozen bipartite edge list
# -----------------------------------------------------------

BIP_EDGES_PATH = "../data/processed/ingredient_recipe_bipartite_edges.csv"

bip_df = pd.read_csv(BIP_EDGES_PATH)

print("Loaded bipartite edges from:", BIP_EDGES_PATH)
print("Rows (edges):", len(bip_df))
print("Columns:", list(bip_df.columns))
display(bip_df.head())

# Safety checks (match the design you already validated)
assert set(bip_df.columns) == {"node_u", "node_v", "weight"}, "Unexpected columns."
assert (bip_df["node_u"].str.startswith("recipe::")).all(), "node_u must be recipe::..."
assert (~bip_df["node_v"].str.startswith("recipe::")).all(), "node_v must be ingredient tokens."
assert (bip_df["weight"] > 0).all(), "Found non-positive weights."

# -----------------------------------------------------------
# A.1 — Build a bipartite NetworkX graph
# -----------------------------------------------------------

B = nx.Graph()

# Add edges: recipe node <-> ingredient node (weight is usually 1)
B.add_weighted_edges_from(
    bip_df[["node_u", "node_v", "weight"]].itertuples(index=False, name=None),
    weight="weight"
)

# Split nodes
recipe_nodes = [n for n in B.nodes if str(n).startswith("recipe::")]
ingredient_nodes = [n for n in B.nodes if not str(n).startswith("recipe::")]

print("\nBipartite graph built.")
print("Recipe nodes:", len(recipe_nodes))
print("Ingredient nodes:", len(ingredient_nodes))
print("Total nodes:", B.number_of_nodes())
print("Edges:", B.number_of_edges())

# -----------------------------------------------------------
# A.2 — (IMPORTANT) Build a 2-step INGREDIENT projection for Node2Vec
# Why: Node2Vec on the raw bipartite graph mixes node types; for substitution we want
# ingredient-ingredient similarity from shared recipe contexts.
#
# We construct a new ingredient graph where:
# - nodes are ingredients
# - edge weight between i and j counts how many recipes contain both (bipartite 2-hop)
# - we do NOT create recipe nodes in this final graph
# -----------------------------------------------------------

from itertools import combinations
from collections import Counter

# Build: recipe -> set(ingredients)
# (Using sets ensures a recipe contributes at most 1 to any ingredient pair.)
recipe_to_ings = {}
for r in recipe_nodes:
    nbrs = B.neighbors(r)
    recipe_to_ings[r] = set(nbrs)

# Count shared-recipe co-membership via 2-hop paths
pair_counts = Counter()
for r, ings in recipe_to_ings.items():
    if len(ings) < 2:
        continue
    # unordered pairs of ingredients in this recipe
    for a, b in combinations(sorted(ings), 2):
        pair_counts[(a, b)] += 1

print("\n2-step ingredient projection built.")
print("Unique ingredient-ingredient edges:", len(pair_counts))

# Convert to DataFrame (counts)
proj_df = pd.DataFrame(
    [{"ingredient_a": a, "ingredient_b": b, "count": c} for (a, b), c in pair_counts.items()]
)

# -----------------------------------------------------------
# A.3 — Degree-normalised weighting for the projected graph
# Motivation: hub ingredients dominate; we normalise by ingredient degrees in the
# bipartite graph (how many recipes each ingredient appears in).
#
# We use a simple bipartite-degree normalisation:
#   w_ij = c_ij / sqrt(deg_i * deg_j)
# where deg_i is ingredient degree in the bipartite graph (recipes containing i).
# This is gentler than assoc-strength and usually keeps common functional edges alive.
# -----------------------------------------------------------

# ingredient degree in bipartite graph = number of recipes containing ingredient
deg_ing = dict(B.degree(ingredient_nodes))

proj_df["deg_a"] = proj_df["ingredient_a"].map(deg_ing).astype(float)
proj_df["deg_b"] = proj_df["ingredient_b"].map(deg_ing).astype(float)

# defensive (should not happen)
proj_df = proj_df.dropna(subset=["deg_a", "deg_b"])

proj_df["weight"] = proj_df["count"] / np.sqrt(proj_df["deg_a"] * proj_df["deg_b"])

# Sort for inspection
proj_df = proj_df.sort_values("weight", ascending=False).reset_index(drop=True)

print("\nTop 10 edges after bipartite-degree normalisation:")
display(proj_df.head(10)[["ingredient_a", "ingredient_b", "count", "deg_a", "deg_b", "weight"]])

# Save this as a frozen input for embeddings (recommended)
OUT_DIR = Path("../data/processed")
PROJ_EDGES_PATH = OUT_DIR / "ingredient_edges_from_bipartite_deg_norm.csv"
proj_df[["ingredient_a", "ingredient_b", "weight"]].to_csv(PROJ_EDGES_PATH, index=False)

print("\nSaved projected ingredient edge list to:", PROJ_EDGES_PATH)
print("Rows saved:", len(proj_df))

# -----------------------------------------------------------
# B.0 — Train Node2Vec on the projected ingredient graph
# Keep your best-performing Node2Vec settings (unweighted/DFS-biased style),
# but we DO use weight_key here because weights are now “hub-mitigated” and smooth.
# -----------------------------------------------------------

G_bip_proj = nx.Graph()
G_bip_proj.add_weighted_edges_from(
    proj_df[["ingredient_a", "ingredient_b", "weight"]].itertuples(index=False, name=None),
    weight="weight"
)

print("\nProjected graph ready for Node2Vec.")
print("Nodes:", G_bip_proj.number_of_nodes())
print("Edges:", G_bip_proj.number_of_edges())

# --- Iteration label for this new graph family ---
ITER = "BIPPROJ_DEGNORM"

DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
WINDOW = 5
P = 1
Q = 4
SEED = 42

node2vec_bip = Node2Vec(
    G_bip_proj,
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    weight_key="weight",
    workers=2,
    seed=SEED
)

model_bip = node2vec_bip.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=1
)

print(f"\nTraining complete: {ITER} (p={P}, q={Q}, window={WINDOW}).")

# -----------------------------------------------------------
# B.1 — Save embeddings
# -----------------------------------------------------------

EMB_DIR = Path("../data/processed/embeddings")
EMB_DIR.mkdir(parents=True, exist_ok=True)

emb_path = EMB_DIR / f"ingredient_node2vec_{ITER}_p{P}_q{Q}_w{WINDOW}.txt"
model_bip.wv.save_word2vec_format(str(emb_path))

print("Saved embeddings to:", emb_path)
print("Number of embeddings saved:", len(model_bip.wv))
print("Embedding dimension:", model_bip.vector_size)

# -----------------------------------------------------------
# C.0 — Evidence: nearest neighbours (reuse your canonical probes + helper)
# -----------------------------------------------------------

print_neighbours(
    model_bip,
    probe_ingredients_canonical,
    topn=10,
    title=f"{ITER} neighbours — ingredient projection from bipartite graph (deg-normalised)"
)


Loaded bipartite edges from: ../data/processed/ingredient_recipe_bipartite_edges.csv
Rows (edges): 1522980
Columns: ['node_u', 'node_v', 'weight']


,node_u,node_v,weight
0,recipe::137739,honey,1
1,recipe::137739,mexican,1
2,recipe::137739,mixed spice,1
3,recipe::137739,winter squash,1
4,recipe::31490,cheese,1



Bipartite graph built.
Recipe nodes: 219890
Ingredient nodes: 2698
Total nodes: 222588
Edges: 1522980

2-step ingredient projection built.
Unique ingredient-ingredient edges: 419913

Top 10 edges after bipartite-degree normalisation:


,ingredient_a,ingredient_b,count,deg_a,deg_b,weight
0,corn chex,wheat chex,37,90.0,50.0,0.551563
1,egg,flour,27623,51352.0,49386.0,0.548517
2,rice chex,wheat chex,40,107.0,50.0,0.546869
3,flour,powder,15622,49386.0,17454.0,0.532093
4,corn chex,rice chex,52,90.0,107.0,0.529895
5,nori,sushi rice,37,103.0,55.0,0.491589
6,flour,soda,12537,49386.0,14049.0,0.475958
7,egg,powder,13475,51352.0,17454.0,0.450093
8,powder,soda,6754,17454.0,14049.0,0.431311
9,tapioca flour,xanthan gum,85,147.0,276.0,0.421993



Saved projected ingredient edge list to: ../data/processed/ingredient_edges_from_bipartite_deg_norm.csv
Rows saved: 419913

Projected graph ready for Node2Vec.
Nodes: 2698
Edges: 419913


Generating walks (CPU: 2): 100%|██████████| 10/10 [00:54<00:00,  5.48s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.ou


Training complete: BIPPROJ_DEGNORM (p=1, q=4, window=5).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_BIPPROJ_DEGNORM_p1_q4_w5.txt
Number of embeddings saved: 2698
Embedding dimension: 128

######################################################################
BIPPROJ_DEGNORM neighbours — ingredient projection from bipartite graph (deg-normalised)
######################################################################
Present probes: ['real butter', 'pure olive oil', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: []

Probe: real butter
01. vanilla caramel                cosine=0.448
02. real vanilla                   cosine=0.411
03. vanilla chip                   cosine=0.410
04. milky way bar                  cosine=0.405
05. sifted flour                   cosine=0.401
06. milk chocolate candy bar       cosine=0.397
07. cris

In [1]:
# ===========================================================
# Z.0 — 10k BEST-QUALITY Bipartite Node2Vec (ingredient↔recipe)
# Direct training on bipartite graph + hub-mitigating weights
# ===========================================================

import pandas as pd
import numpy as np
import networkx as nx
from node2vec import Node2Vec
from pathlib import Path

# -----------------------------
# Z.1 — Load / build 10k bipartite edges
# -----------------------------
# Preferred: a pre-built 10k bipartite edge list
BIP_10K_PATH = Path("../data/processed/ingredient_recipe_bipartite_edges_10k.csv")

# Fallback: sample 10k recipes from the full bipartite edge list if the 10k file doesn't exist
BIP_FULL_PATH = Path("../data/processed/ingredient_recipe_bipartite_edges.csv")

if BIP_10K_PATH.exists():
    bip_df = pd.read_csv(BIP_10K_PATH)
    print("Loaded 10k bipartite edges from:", BIP_10K_PATH)
else:
    if not BIP_FULL_PATH.exists():
        raise FileNotFoundError(
            "Neither 10k nor full bipartite edge list found.\n"
            f"Expected one of:\n- {BIP_10K_PATH}\n- {BIP_FULL_PATH}"
        )
    full_df = pd.read_csv(BIP_FULL_PATH)
    print("Loaded FULL bipartite edges from:", BIP_FULL_PATH)

    # Expect node_u is recipe::..., node_v is ingredient token
    if "node_u" not in full_df.columns or "node_v" not in full_df.columns:
        raise ValueError("Expected columns: node_u, node_v (and optional weight)")

    # Sample 10k distinct recipes, then keep their edges
    recipes = full_df["node_u"].astype(str).unique()
    if len(recipes) < 10_000:
        print(f"WARNING: only {len(recipes)} unique recipes available; using all of them.")
        chosen = set(recipes)
    else:
        rng = np.random.default_rng(42)
        chosen = set(rng.choice(recipes, size=10_000, replace=False))

    bip_df = full_df[full_df["node_u"].astype(str).isin(chosen)].copy()
    print("Constructed 10k subset from full file.")
    print("Edges kept:", len(bip_df))
    print("Unique recipes kept:", bip_df["node_u"].nunique())

# Standardize schema (ignore any existing 'weight' column — we will compute better weights)
bip_df = bip_df[["node_u", "node_v"]].copy()
bip_df["node_u"] = bip_df["node_u"].astype(str)
bip_df["node_v"] = bip_df["node_v"].astype(str)

assert bip_df["node_u"].str.startswith("recipe::").all(), "node_u must be recipe::..."
assert (~bip_df["node_v"].str.startswith("recipe::")).all(), "node_v must be ingredient tokens."

# -----------------------------
# Z.2 — Hub-mitigating edge weights: 1/deg(recipe)
# -----------------------------
# Idea: every recipe contributes a total weight of ~1 distributed across its ingredients.
# This prevents huge recipes (30+ ingredients) from dominating training.
recipe_deg = bip_df.groupby("node_u").size().to_dict()
bip_df["weight"] = bip_df["node_u"].map(recipe_deg).astype(float)
bip_df["weight"] = 1.0 / bip_df["weight"]

print("\n10k bipartite weight scheme:")
print("Recipe degree: min =", min(recipe_deg.values()), "max =", max(recipe_deg.values()))
print("Edge weight:   min =", bip_df["weight"].min(), "max =", bip_df["weight"].max())

# -----------------------------
# Z.3 — Build weighted bipartite graph
# -----------------------------
B = nx.Graph()
B.add_weighted_edges_from(
    bip_df[["node_u", "node_v", "weight"]].itertuples(index=False, name=None),
    weight="weight"
)

recipe_nodes = [n for n in B.nodes if str(n).startswith("recipe::")]
ingredient_nodes = [n for n in B.nodes if not str(n).startswith("recipe::")]

print("\nBipartite graph built (10k).")
print("Recipe nodes:", len(recipe_nodes))
print("Ingredient nodes:", len(ingredient_nodes))
print("Total nodes:", B.number_of_nodes())
print("Edges:", B.number_of_edges())

# -----------------------------
# Z.4 — Train Node2Vec directly on the bipartite graph
# -----------------------------
# “Role similarity” bias: higher q encourages exploring outward (DFS-ish),
# which helps “structural equivalence” (substitution-like) more than pure homophily.
DIM = 128
WALK_LEN = 80      # slightly longer helps in bipartite (alternating types)
NUM_WALKS = 15     # keep modest for 10k (faster, still decent)
WINDOW = 5         # smaller window reduces “everything becomes similar”
P = 1
Q = 4
SEED = 42

node2vec_bip_10k = Node2Vec(
    B,
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    weight_key="weight",   # IMPORTANT: use hub-mitigating weights in transitions
    workers=4,
    seed=SEED
)

model_bip_10k = node2vec_bip_10k.fit(
    window=WINDOW,
    min_count=2,           # drop ultra-rare tokens from training dynamics
    batch_words=128,
    workers=2
)

print("\nTraining complete: BIPARTITE_10K (weighted by 1/deg(recipe)).")
print("Total vectors learned (recipes + ingredients):", len(model_bip_10k.wv))
print("Embedding dimension:", model_bip_10k.vector_size)

# -----------------------------
# Z.5 — Save ingredient-only embeddings
# -----------------------------
EMB_DIR = Path("../data/processed/embeddings")
EMB_DIR.mkdir(parents=True, exist_ok=True)

ITER_TAG = f"BIPARTITE_10K_rdegW_p{P}_q{Q}_w{WINDOW}_l{WALK_LEN}_d{DIM}"
ing_only_path = EMB_DIR / f"ingredient_node2vec_{ITER_TAG}_ING_ONLY.txt"

ingredient_vocab = [t for t in model_bip_10k.wv.index_to_key if not str(t).startswith("recipe::")]
dim = model_bip_10k.vector_size

with open(ing_only_path, "w", encoding="utf-8") as f:
    f.write(f"{len(ingredient_vocab)} {dim}\n")
    for token in ingredient_vocab:
        vec = model_bip_10k.wv[token]
        f.write(token + " " + " ".join(f"{x:.6f}" for x in vec) + "\n")

print("Saved ingredient-only embeddings to:", ing_only_path)
print("Ingredient-only vectors:", len(ingredient_vocab))

# -----------------------------
# Z.6 — Evidence: neighbours for your canonical probes
# -----------------------------
def print_neighbours(model, probe_list, topn=10, title=""):
    vocab = set(model.wv.key_to_index.keys())
    present = [x for x in probe_list if x in vocab]
    missing = [x for x in probe_list if x not in vocab]

    if title:
        print("\n" + "#" * 70)
        print(title)
        print("#" * 70)

    print("Present probes:", present)
    print("Missing probes:", missing)

    for ing in present:
        print("\n" + "=" * 60)
        print(f"Probe: {ing}")
        for i, (nbr, sim) in enumerate(model.wv.most_similar(ing, topn=topn), start=1):
            # Hide recipe nodes in neighbour list (we only care ingredient neighbours)
            if str(nbr).startswith("recipe::"):
                continue
            print(f"{i:02d}. {nbr:<30} cosine={sim:.3f}")

# Reuse your canonical probe list if it exists, otherwise define it
try:
    probe_ingredients_canonical
except NameError:
    probe_ingredients_canonical = [
        "real butter", "pure olive oil", "canola oil",
        "milk", "yogurt", "cream",
        "flour", "bread flour",
        "cane sugar", "light-brown sugar", "honey",
        "garlic", "red onion",
        "chicken", "beef",
        "rice", "pasta"
    ]

print_neighbours(
    model_bip_10k,
    probe_ingredients_canonical,
    topn=20,
    title="BIPARTITE 10K neighbours (ingredient-only view; recipe hubs downweighted)"
)


/Users/adarpalaoglu/Documents/cs310-safeplate/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded FULL bipartite edges from: ../data/processed/ingredient_recipe_bipartite_edges.csv
Constructed 10k subset from full file.
Edges kept: 69367
Unique recipes kept: 10000

10k bipartite weight scheme:
Recipe degree: min = 3 max = 24
Edge weight:   min = 0.041666666666666664 max = 0.3333333333333333

Bipartite graph built (10k).
Recipe nodes: 10000
Ingredient nodes: 2425
Total nodes: 12425
Edges: 69367


Generating walks (CPU: 3): 100%|██████████| 4/4 [00:44<00:00, 11.07s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_


Training complete: BIPARTITE_10K (weighted by 1/deg(recipe)).
Total vectors learned (recipes + ingredients): 12425
Embedding dimension: 128
Saved ingredient-only embeddings to: ../data/processed/embeddings/ingredient_node2vec_BIPARTITE_10K_rdegW_p1_q4_w5_l80_d128_ING_ONLY.txt
Ingredient-only vectors: 2425

######################################################################
BIPARTITE 10K neighbours (ingredient-only view; recipe hubs downweighted)
######################################################################
Present probes: ['real butter', 'canola oil', 'milk', 'yogurt', 'cream', 'flour', 'bread flour', 'cane sugar', 'light-brown sugar', 'honey', 'garlic', 'red onion', 'chicken', 'beef', 'rice', 'pasta']
Missing probes: ['pure olive oil']

Probe: real butter
02. real vanilla extract           cosine=0.754
07. bailey irish cream             cosine=0.614
08. caramel ice cream topping      cosine=0.562
16. chai tea                       cosine=0.529
17. bread dough             

In [1]:
# ===========================================================
# K.0 — (RESEARCH-RECOMMENDED) Node2Vec on the LOSSLESS
# INGREDIENT–RECIPE BIPARTITE GRAPH (NO PROJECTION)
#
# Key idea:
# - Train Node2Vec directly on the bipartite graph B (ingredient <-> recipe edges).
# - Walks naturally alternate: ingredient -> recipe -> ingredient -> ...
# - Keep edges UNWEIGHTED (weight=1, and we do NOT pass weight_key).
# - For substitution, we ONLY evaluate + save embeddings for ingredient nodes.
# ===========================================================

import pandas as pd
import networkx as nx
from node2vec import Node2Vec
from pathlib import Path

# -----------------------------------------------------------
# K.1 — Load the frozen bipartite edge list (read-only)
# -----------------------------------------------------------

BIP_EDGES_PATH = "../data/processed/ingredient_recipe_bipartite_edges.csv"
bip_df = pd.read_csv(BIP_EDGES_PATH)

print("Loaded bipartite edges from:", BIP_EDGES_PATH)
print("Rows (edges):", len(bip_df))
print("Columns:", list(bip_df.columns))
display(bip_df.head())

# Safety checks (match the bipartite design)
assert set(bip_df.columns) == {"node_u", "node_v", "weight"}, "Unexpected columns."
assert (bip_df["node_u"].astype(str).str.startswith("recipe::")).all(), "node_u must be recipe::..."
assert (~bip_df["node_v"].astype(str).str.startswith("recipe::")).all(), "node_v must be ingredient tokens."
assert (bip_df["weight"] > 0).all(), "Found non-positive weights."
assert (bip_df["weight"] == 1).all(), "Expected all bipartite weights to be 1."

# -----------------------------------------------------------
# K.2 — Build the bipartite graph B (UNWEIGHTED)
# -----------------------------------------------------------

B = nx.Graph()
B.add_edges_from(bip_df[["node_u", "node_v"]].itertuples(index=False, name=None))

recipe_nodes = [n for n in B.nodes if str(n).startswith("recipe::")]
ingredient_nodes = [n for n in B.nodes if not str(n).startswith("recipe::")]

print("\nBipartite graph built (UNWEIGHTED).")
print("Recipe nodes:", len(recipe_nodes))
print("Ingredient nodes:", len(ingredient_nodes))
print("Total nodes:", B.number_of_nodes())
print("Edges:", B.number_of_edges())

# -----------------------------------------------------------
# K.3 — Train Node2Vec on B (research-style, unweighted)
# ONLY CHANGE vs previous slow run: workers increased to 8
# -----------------------------------------------------------

ITER = "BIPARTITE_DIRECT_UNWEIGHTED"
DIM = 128
WALK_LEN = 60
NUM_WALKS = 20
WINDOW = 5
P = 1
Q = 4
SEED = 42

node2vec_bip_direct = Node2Vec(
    B,
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    # IMPORTANT: unweighted (no weight_key)
    workers=8,     # <-- ONLY intentional change (speed-up)
    seed=SEED
)

model_bip_direct = node2vec_bip_direct.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=8     # <-- match CPU usage during training
)

print(f"\nTraining complete: {ITER} (UNWEIGHTED, p={P}, q={Q}, window={WINDOW}).")
print("Total embeddings learned (ingredients + recipes):", len(model_bip_direct.wv))
print("Embedding dimension:", model_bip_direct.vector_size)

# -----------------------------------------------------------
# K.4 — Save embeddings
# -----------------------------------------------------------

EMB_DIR = Path("../data/processed/embeddings")
EMB_DIR.mkdir(parents=True, exist_ok=True)

full_path = EMB_DIR / f"node2vec_{ITER}_FULL_p{P}_q{Q}_w{WINDOW}_l{WALK_LEN}.txt"
ing_only_path = EMB_DIR / f"node2vec_{ITER}_INGREDIENTS_ONLY_p{P}_q{Q}_w{WINDOW}_l{WALK_LEN}.txt"

model_bip_direct.wv.save_word2vec_format(str(full_path))
print("Saved FULL embeddings to:", full_path)

dim = model_bip_direct.vector_size
ingredient_vocab = [n for n in model_bip_direct.wv.index_to_key if not str(n).startswith("recipe::")]

with open(ing_only_path, "w", encoding="utf-8") as f:
    f.write(f"{len(ingredient_vocab)} {dim}\n")
    for token in ingredient_vocab:
        vec = model_bip_direct.wv[token]
        f.write(token + " " + " ".join(f"{x:.6f}" for x in vec) + "\n")

print("Saved INGREDIENT-ONLY embeddings to:", ing_only_path)
print("Ingredient-only vectors:", len(ingredient_vocab))


/Users/adarpalaoglu/Documents/cs310-safeplate/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded bipartite edges from: ../data/processed/ingredient_recipe_bipartite_edges.csv
Rows (edges): 1522980
Columns: ['node_u', 'node_v', 'weight']


,node_u,node_v,weight
0,recipe::137739,honey,1
1,recipe::137739,mexican,1
2,recipe::137739,mixed spice,1
3,recipe::137739,winter squash,1
4,recipe::31490,cheese,1



Bipartite graph built (UNWEIGHTED).
Recipe nodes: 219890
Ingredient nodes: 2698
Total nodes: 222588
Edges: 1522980


Computing transition probabilities: 100%|██████████| 222588/222588 [4:05:41<00:00, 15.10it/s]     


: 

## 📌 Experiment Plan (Co-occurrence vs Bipartite Node2Vec) + Frozen Evaluation Protocol

This notebook is part of **SafePlate: Ingredient Triggers and Ingredient Substitution**.

### What we are doing (overall methodology)

Our final goal is to decide which embedding approach produces the **best ingredient substitution behaviour**.

We will train Node2Vec embeddings on **two different graph constructions**:

1. **Ingredient–ingredient co-occurrence graph** (fast to train → tuned first)
2. **Ingredient–recipe bipartite graph** (slower → tuned after co-occurrence)

The process is deliberately **two-stage** to ensure a fair comparison:

1. **Within-family optimisation (co-occurrence):**  
   Tune Node2Vec parameters and graph weighting/pruning variants on the co-occurrence graph until improvements plateau under a fixed automated metric.

2. **Within-family optimisation (bipartite):**  
   Repeat the same tuning process on the bipartite-based approach.

3. **Final cross-family comparison:**  
   Compare the *best co-occurrence model* vs the *best bipartite model* using the **same frozen metric and evaluation protocol**, and select the overall best embedding model for the substitution engine.

This avoids an unfair outcome where one graph family appears worse simply because it was under-tuned.

---

## ✅ Frozen Constants (do not change mid-experiment)

### Recommendation setting
- **Top-k = 10** (primary reporting).
- Optional sanity reporting: **Top-k = 5**.

### Evaluation ingredient set (fixed across all models)
- Use a fixed set of **300 query ingredients**, sampled once and reused across all runs.
- Constructed by stratified frequency sampling:
  - 100 from **head** frequency band (top ~5%)
  - 100 from **mid** band (5–30%)
  - 100 from **tail** band (30–70%, excluding ultra-rare noise)
- Sampling seed is frozen (e.g., **seed = 42**).
- Exclude junk tokens and (optionally) ultra-rare ingredients (e.g., frequency < 10).

### Candidate filtering (applied to all embedding models)
When generating top-k neighbours for a query ingredient:
- exclude the ingredient itself
- exclude `recipe::...` nodes (for bipartite-trained embeddings)
- optionally exclude candidates with very low frequency (e.g., < 10) for stability

---

## ✅ Frozen Automated Metric (used for tuning + final model selection)

This metric is designed to measure **substitutability**, not raw co-occurrence frequency.

**Context representation (reference, fixed):**
- Build **PPMI-based ingredient context vectors** from the corpus co-occurrence statistics.

**Exclusivity penalty (discourages "used together" items):**
- Define `coUse(i, j) = max(P(j|i), P(i|j))` where `P(j|i) = C(i,j) / freq(i)`.

**PairScore (for a recommended neighbour j of query i):**
- `PairScore(i, j) = cosine(PPMI(i), PPMI(j)) × (1 − coUse(i, j))`

**Metric@k for a query ingredient i:**
- `Metric@k(i) = mean_{j in topk(i)} PairScore(i, j)`

**Overall model score:**
- average `Metric@k(i)` over the fixed set of 300 evaluation ingredients

Notes:
- The metric **does not generate candidates**; it only scores model outputs.
- The metric **cannot be trivially won** by ranking by co-occurrence counts.

---

## ✅ Guardrails for Fair Optimisation (time-feasible version)

### Randomness control (single-seed during tuning)
Due to compute constraints, **within-family tuning uses a single fixed seed**:
- training seed: **42**
- the same seed is reused for every configuration within a graph family

This keeps the tuning process consistent while staying time-feasible.

### Multi-seed only for the final comparison (robustness check)
To ensure the *final* decision is not due to randomness, we run **3 seeds only once**:
- seeds: `{42, 43, 44}`
- applied only to:
  - the best co-occurrence configuration
  - the best bipartite configuration

The final choice is made using:
- **mean score across seeds** (primary)
- **std dev across seeds** (stability reporting)

### Accepting improvements during tuning (single-seed rule)
During tuning (seed=42 only), a configuration is considered better if:
- it improves the model score by **≥ +1% (relative)** over the current best

(We intentionally avoid seed-based win rules during tuning to save time.)

### Tuning discipline
- Change **one thing at a time** between iterations (p, q, window, walks, weighting, pruning, etc.).
- Keep everything else fixed.

### Stopping rule (plateau)
Stop tuning a graph family if:
- no improvement (≥ +1% relative) occurs across the last **5 configurations**

### Budget
- Maximum **20 configurations per graph family**
- Multi-seed evaluation is performed **only** for the final two best models

---

## 📎 Evidence kept per iteration (for write-up)
For each iteration we record:
- the full configuration (graph version + Node2Vec hyperparameters)
- the automated metric score (seed=42 during tuning)
- neighbours for the fixed probe list (qualitative sanity)
- short notes: what improved / got worse / next hypothesis

For the final comparison (best vs best), we additionally record:
- mean and std across seeds `{42, 43, 44}`

This creates a clear experimental trail for the dissertation: decisions are traceable, repeatable, and fair within time constraints.


In [5]:
# ===========================================================
# METRIC + RUNNER (rank-aware, robust):
# - loads edges_df + builds PPMI tables,
# - builds/loads eval set (frozen at n_total=300),
# - loads Iteration 4 embeddings even if model_iter4 not in memory,
# - scores top-k lists using a RANK-WEIGHTED aggregation.
#
# FIXES INCLUDED:
# - Handles ingredient tokens WITH SPACES (e.g., "real butter") when loading
#   Word2Vec text embeddings (gensim default loader breaks on spaces).
# - Robust eval set creation that truly freezes at n_total=300.
# - Safe handling of OOV tokens + empty candidate lists.
#
# NEW:
# - Ranking affects score via DCG-style log discount (or reciprocal).
#   Earlier ranks contribute more than later ranks.
# ===========================================================

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import coo_matrix
from gensim.models import KeyedVectors

# -----------------------------------------------------------
# 0) Paths (adjust only if your filenames differ)
# -----------------------------------------------------------
EDGES_PATH = Path("../data/processed/ingredient_cooccurrence_edges.csv")

# This should match the save name from your Iteration 4 cell
ITER4_EMB_PATH = Path("../data/processed/embeddings/ingredient_node2vec_iter4_UNWEIGHTED_p1_q4_w5.txt")

# Frozen evaluation ingredient set
EVAL_PATH = Path("../data/processed/eval_ingredients_300_seed42.csv")


# -----------------------------------------------------------
# 1) Load edges_df for metric reference
# -----------------------------------------------------------
edges_df = pd.read_csv(EDGES_PATH)
assert {"ingredient_a", "ingredient_b", "weight"} <= set(edges_df.columns), \
    "Expected columns: ingredient_a, ingredient_b, weight"

print("Loaded edges_df for metric reference.")
print("Rows:", len(edges_df))
print("Columns:", list(edges_df.columns))


# -----------------------------------------------------------
# 2) Build PPMI + counts tables
# -----------------------------------------------------------
def build_ppmi_and_counts(edges_df: pd.DataFrame):
    df = edges_df.copy()
    df["ingredient_a"] = df["ingredient_a"].astype(str)
    df["ingredient_b"] = df["ingredient_b"].astype(str)
    df["weight"] = df["weight"].astype(float)

    vocab = pd.Index(pd.unique(pd.concat([df["ingredient_a"], df["ingredient_b"]], ignore_index=True)))
    idx = {t: i for i, t in enumerate(vocab)}
    n = len(vocab)

    ia = df["ingredient_a"].map(idx).to_numpy()
    ib = df["ingredient_b"].map(idx).to_numpy()
    w  = df["weight"].to_numpy()

    # Symmetrize counts (undirected)
    rows = np.concatenate([ia, ib])
    cols = np.concatenate([ib, ia])
    data = np.concatenate([w,  w ])

    C = coo_matrix((data, (rows, cols)), shape=(n, n)).tocsr()
    C.sum_duplicates()

    marg = np.array(C.sum(axis=1)).ravel()
    total = float(marg.sum())

    C_coo = C.tocoo()
    ii = C_coo.row
    jj = C_coo.col
    cij = C_coo.data

    denom = (marg[ii] * marg[jj])
    valid = (cij > 0) & (denom > 0)

    ii = ii[valid]
    jj = jj[valid]
    cij = cij[valid]
    denom = denom[valid]

    # PMI(i,j) = log( (c_ij * total) / (marg_i * marg_j) )
    pmi = np.log((cij * total) / denom)
    ppmi = np.maximum(pmi, 0.0)

    P = coo_matrix((ppmi, (ii, jj)), shape=(n, n)).tocsr()
    P.sum_duplicates()

    row_norms = np.sqrt(P.multiply(P).sum(axis=1)).A1
    row_norms[row_norms == 0] = 1e-12

    return vocab, idx, C, marg, total, P, row_norms


vocab_ppmi, idx_ppmi, C_counts, marg_counts, total_mass, P_ppmi, ppmi_norms = build_ppmi_and_counts(edges_df)

print("\nPPMI metric tables built.")
print("Vocab size:", len(vocab_ppmi))
print("Nonzeros in PPMI matrix:", P_ppmi.nnz)


# -----------------------------------------------------------
# 3) PairScore = PPMI cosine × (1 - co-use)
#   co-use(i,j) = max(P(j|i), P(i|j)), where P(j|i)=c_ij/marg_i
# -----------------------------------------------------------
def ppmi_cosine(i_token: str, j_token: str) -> float:
    if i_token not in idx_ppmi or j_token not in idx_ppmi:
        return 0.0
    i = idx_ppmi[i_token]
    j = idx_ppmi[j_token]
    dot = P_ppmi[i].multiply(P_ppmi[j]).sum()
    return float(dot / (ppmi_norms[i] * ppmi_norms[j]))


def co_use(i_token: str, j_token: str) -> float:
    if i_token not in idx_ppmi or j_token not in idx_ppmi:
        return 0.0
    i = idx_ppmi[i_token]
    j = idx_ppmi[j_token]

    cij = float(C_counts[i, j])
    if cij <= 0:
        return 0.0

    mi = float(marg_counts[i])
    mj = float(marg_counts[j])
    if mi <= 0 or mj <= 0:
        return 0.0

    return float(max(cij / mi, cij / mj))


def pair_score(i_token: str, j_token: str) -> float:
    sim = ppmi_cosine(i_token, j_token)
    penalty = 1.0 - co_use(i_token, j_token)
    penalty = max(0.0, min(1.0, penalty))
    return float(sim * penalty)


# -----------------------------------------------------------
# 4) Build / load eval set (ensure n_total=300 exactly)
# -----------------------------------------------------------
def build_eval_set(vocab_index: pd.Index, marg: np.ndarray, seed=42, n_total=300):
    rng = np.random.default_rng(seed)

    order = np.argsort(-marg)  # descending by marginal mass
    tokens_sorted = vocab_index[order].to_numpy()

    # Keep top 70% only (avoid extreme tail noise)
    cutoff = int(len(tokens_sorted) * 0.70)
    tokens_70 = tokens_sorted[:cutoff]

    n = len(tokens_70)
    head_end = max(1, int(0.05 * n))
    mid_end  = max(head_end + 1, int(0.30 * n))

    head = tokens_70[:head_end]
    mid  = tokens_70[head_end:mid_end]
    tail = tokens_70[mid_end:]

    def sample(arr, k):
        if len(arr) <= k:
            return list(arr)
        return list(rng.choice(arr, size=k, replace=False))

    # Start with 100/100/100 stratified sample (then de-dup and top-up)
    eval_tokens = sample(head, 100) + sample(mid, 100) + sample(tail, 100)

    # De-dup preserving order
    eval_tokens = list(dict.fromkeys(eval_tokens))

    # Top-up if short
    if len(eval_tokens) < n_total:
        pool = list(tokens_70)
        rng.shuffle(pool)
        have = set(eval_tokens)
        for t in pool:
            if t not in have:
                eval_tokens.append(t)
                have.add(t)
            if len(eval_tokens) >= n_total:
                break

    # Trim if long
    eval_tokens = eval_tokens[:n_total]

    # Deterministic shuffle (given seed)
    rng.shuffle(eval_tokens)
    return eval_tokens


def load_or_build_eval_set(path: Path, vocab_index: pd.Index, marg: np.ndarray, seed=42, n_total=300):
    if path.exists():
        df = pd.read_csv(path)
        if "ingredient" not in df.columns:
            raise ValueError(f"Eval file exists but has no 'ingredient' column: {path}")
        tokens = df["ingredient"].astype(str).tolist()

        # If it isn't exactly n_total, rebuild deterministically and overwrite
        if len(tokens) != n_total:
            tokens = build_eval_set(vocab_index, marg, seed=seed, n_total=n_total)
            pd.DataFrame({"ingredient": tokens}).to_csv(path, index=False)
            print(f"\nLoaded eval set but it was not {n_total}; rebuilt & re-saved:", path, "| n =", len(tokens))
        else:
            print("\nLoaded frozen eval set:", path, "| n =", len(tokens))
        return tokens

    tokens = build_eval_set(vocab_index, marg, seed=seed, n_total=n_total)
    pd.DataFrame({"ingredient": tokens}).to_csv(path, index=False)
    print("\nSaved frozen eval set:", path, "| n =", len(tokens))
    return tokens


eval_ingredients = load_or_build_eval_set(EVAL_PATH, vocab_ppmi, marg_counts, seed=42, n_total=300)


# -----------------------------------------------------------
# 5) RANK WEIGHTS (so rank 1 counts more than rank k)
# -----------------------------------------------------------
def rank_weights(k: int, scheme: str = "log") -> np.ndarray:
    """
    Returns normalized weights (sum to 1) for ranks 1..k.
    - scheme="log": DCG-style discount 1/log2(r+1)
    - scheme="reciprocal": 1/r
    """
    r = np.arange(1, k + 1, dtype=float)
    if scheme == "log":
        w = 1.0 / np.log2(r + 1.0)
    elif scheme == "reciprocal":
        w = 1.0 / r
    else:
        raise ValueError("scheme must be 'log' or 'reciprocal'")
    return w / w.sum()


def aggregate_ranked(scores, scheme: str = "log") -> float:
    """
    Rank-weighted aggregation for a list of per-rank scores.
    """
    scores = np.asarray(scores, dtype=float)
    if scores.size == 0:
        return 0.0
    w = rank_weights(len(scores), scheme=scheme)
    return float(np.sum(w * scores))


# -----------------------------------------------------------
# 6) Score a model (works with KeyedVectors or gensim model.wv)
#    NEW: uses rank-weighted aggregation
# -----------------------------------------------------------
def metric_score_kv(
    kv,
    eval_tokens,
    topk=10,
    exclude_prefixes=("recipe::",),
    min_candidate_marg=0.0,
    rank_scheme="log",       # NEW: "log" (DCG) or "reciprocal"
):
    model_vocab = set(kv.key_to_index.keys())
    marg_lookup = {t: float(marg_counts[idx_ppmi[t]]) for t in model_vocab if t in idx_ppmi}

    per_query_scores = []
    missing_queries = 0
    empty_lists = 0

    for q in eval_tokens:
        if q not in model_vocab or q not in idx_ppmi:
            missing_queries += 1
            continue

        # ask for more than topk in case filtering removes many
        raw = kv.most_similar(q, topn=topk * 5)

        cands = []
        for cand, _sim in raw:
            if cand == q:
                continue
            if any(str(cand).startswith(p) for p in exclude_prefixes):
                continue
            if cand not in idx_ppmi:
                continue
            if min_candidate_marg > 0 and marg_lookup.get(cand, 0.0) < min_candidate_marg:
                continue
            cands.append(cand)
            if len(cands) >= topk:
                break

        if len(cands) == 0:
            empty_lists += 1
            continue

        # scores are ordered by model rank already (most_similar returns sorted)
        scores = [pair_score(q, c) for c in cands]

        # NEW: rank-aware aggregation instead of mean()
        per_query_scores.append(aggregate_ranked(scores, scheme=rank_scheme))

    overall = float(np.mean(per_query_scores)) if per_query_scores else 0.0
    meta = {
        "missing_queries": missing_queries,
        "empty_lists": empty_lists,
        "scored_queries": len(per_query_scores),
        "total_eval_tokens": len(eval_tokens),
        "rank_scheme": rank_scheme,
    }
    return overall, per_query_scores, meta


# -----------------------------------------------------------
# 7) Load KeyedVectors robustly when tokens may contain spaces
# -----------------------------------------------------------
def load_word2vec_with_spaces(path: Path, encoding="utf-8"):
    """
    Loads a Word2Vec *text* format file where tokens may contain spaces.
    Expected file structure:
      first line: "<n_vocab> <dim>"
      each next line: "<token possibly with spaces> <f1> <f2> ... <fdim>"
    We parse by taking the LAST <dim> fields as floats; everything before is the token.
    """
    if not path.exists():
        raise FileNotFoundError(f"Embeddings file not found: {path}")

    with path.open("r", encoding=encoding, errors="replace") as f:
        header = f.readline().strip().split()
        if len(header) < 2:
            raise ValueError(f"Bad header in embeddings file: {path}")

        n_vocab = int(header[0])
        dim = int(header[1])

        kv = KeyedVectors(vector_size=dim)

        tokens = []
        vectors = np.zeros((n_vocab, dim), dtype=np.float32)

        row = 0
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            parts = line.split(" ")
            if len(parts) < dim + 1:
                continue

            vec_str = parts[-dim:]
            tok_parts = parts[:-dim]
            token = " ".join(tok_parts).strip()

            try:
                vec = np.asarray(vec_str, dtype=np.float32)
            except ValueError:
                continue

            if vec.shape[0] != dim or token == "":
                continue

            if row >= vectors.shape[0]:
                # In case header n_vocab is wrong, grow
                vectors = np.vstack([vectors, np.zeros((1024, dim), dtype=np.float32)])

            tokens.append(token)
            vectors[row, :] = vec
            row += 1

        vectors = vectors[:row, :]
        tokens = tokens[:row]

        kv.add_vectors(tokens, vectors)

    return kv


# -----------------------------------------------------------
# 8) Get Iteration 4 vectors (prefer in-memory, else load from disk)
# -----------------------------------------------------------
def get_iter4_keyedvectors():
    if "model_iter4" in globals():
        print("\nUsing in-memory model_iter4.")
        return globals()["model_iter4"].wv

    if not ITER4_EMB_PATH.exists():
        raise FileNotFoundError(
            "model_iter4 not found in memory AND embeddings file not found.\n"
            f"Expected embeddings at: {ITER4_EMB_PATH}\n"
            "Fix: run your Iteration 4 training cell, or check the filename/path."
        )

    print("\nmodel_iter4 not in memory — loading embeddings from:", ITER4_EMB_PATH)
    kv = load_word2vec_with_spaces(ITER4_EMB_PATH)
    print("Loaded vectors:", len(kv.key_to_index), "| dim:", kv.vector_size)
    return kv


# -----------------------------------------------------------
# 9) Run metric for Iteration 4 (rank-aware)
# -----------------------------------------------------------
TOPK = 10
RANK_SCHEME = "log"   # "log" (DCG-style) or "reciprocal"

kv_iter4 = get_iter4_keyedvectors()

score_iter4, per_q_iter4, meta_iter4 = metric_score_kv(
    kv_iter4,
    eval_ingredients,
    topk=TOPK,
    exclude_prefixes=("recipe::",),
    min_candidate_marg=0.0,
    rank_scheme=RANK_SCHEME
)

print("\n" + "=" * 70)
print(f"METRIC RESULT — Co-occurrence Node2Vec Iteration 4 (UNWEIGHTED, q=4) | rank={RANK_SCHEME}")
print(f"Eval set size: {meta_iter4['total_eval_tokens']}")
print(f"Scored queries: {meta_iter4['scored_queries']}")
print(f"Missing queries (not in model vocab or not in PPMI vocab): {meta_iter4['missing_queries']}")
print(f"Queries with empty candidate list after filtering: {meta_iter4['empty_lists']}")
print(f"Top-k: {TOPK}")
print(f"ModelScore: {score_iter4:.6f}")
print("=" * 70)

if per_q_iter4:
    arr = np.array(per_q_iter4)
    print("\nPer-query Metric@k summary:")
    for p in [0, 10, 25, 50, 75, 90, 100]:
        print(f"  p{p:>3}: {np.percentile(arr, p):.6f}")


Loaded edges_df for metric reference.
Rows: 419913
Columns: ['ingredient_a', 'ingredient_b', 'weight']

PPMI metric tables built.
Vocab size: 2698
Nonzeros in PPMI matrix: 839826

Loaded frozen eval set: ../data/processed/eval_ingredients_300_seed42.csv | n = 300

model_iter4 not in memory — loading embeddings from: ../data/processed/embeddings/ingredient_node2vec_iter4_UNWEIGHTED_p1_q4_w5.txt
Loaded vectors: 2698 | dim: 128

METRIC RESULT — Co-occurrence Node2Vec Iteration 4 (UNWEIGHTED, q=4) | rank=log
Eval set size: 300
Scored queries: 300
Missing queries (not in model vocab or not in PPMI vocab): 0
Queries with empty candidate list after filtering: 0
Top-k: 10
ModelScore: 0.245978

Per-query Metric@k summary:
  p  0: 0.085167
  p 10: 0.149580
  p 25: 0.183105
  p 50: 0.239617
  p 75: 0.298885
  p 90: 0.349402
  p100: 0.503307


In [2]:
# ===========================================================
# METRIC MODULE (define once): build tables once, score any kv
# ===========================================================

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import coo_matrix

class CoocPPMIRankMetric:
    """
    Distributional Context Similarity × Co-use Penalty, rank-aware.
    Build once from edges_df; then call .score(kv) for any embedding model.
    """

    def __init__(
        self,
        edges_path: Path,
        eval_path: Path,
        seed: int = 42,
        n_eval: int = 300,
        keep_top_frac: float = 0.70,
    ):
        self.edges_path = Path(edges_path)
        self.eval_path = Path(eval_path)
        self.seed = seed
        self.n_eval = n_eval
        self.keep_top_frac = keep_top_frac

        self.edges_df = None
        self.vocab_ppmi = None
        self.idx_ppmi = None
        self.C_counts = None
        self.marg_counts = None
        self.total_mass = None
        self.P_ppmi = None
        self.ppmi_norms = None
        self.eval_ingredients = None

    # ---------- Build / load everything once ----------
    def build(self, verbose=True):
        self._load_edges(verbose=verbose)
        self._build_ppmi_and_counts(verbose=verbose)
        self.eval_ingredients = self._load_or_build_eval_set(verbose=verbose)
        return self

    def _load_edges(self, verbose=True):
        df = pd.read_csv(self.edges_path)
        if not {"ingredient_a", "ingredient_b", "weight"} <= set(df.columns):
            raise ValueError("edges_df must have: ingredient_a, ingredient_b, weight")
        self.edges_df = df
        if verbose:
            print("Loaded edges_df for metric reference.")
            print("Rows:", len(df))
            print("Columns:", list(df.columns))

    def _build_ppmi_and_counts(self, verbose=True):
        df = self.edges_df.copy()
        df["ingredient_a"] = df["ingredient_a"].astype(str)
        df["ingredient_b"] = df["ingredient_b"].astype(str)
        df["weight"] = df["weight"].astype(float)

        vocab = pd.Index(pd.unique(pd.concat([df["ingredient_a"], df["ingredient_b"]], ignore_index=True)))
        idx = {t: i for i, t in enumerate(vocab)}
        n = len(vocab)

        ia = df["ingredient_a"].map(idx).to_numpy()
        ib = df["ingredient_b"].map(idx).to_numpy()
        w  = df["weight"].to_numpy()

        # Symmetrize counts
        rows = np.concatenate([ia, ib])
        cols = np.concatenate([ib, ia])
        data = np.concatenate([w,  w ])

        C = coo_matrix((data, (rows, cols)), shape=(n, n)).tocsr()
        C.sum_duplicates()

        marg = np.array(C.sum(axis=1)).ravel()
        total = float(marg.sum())

        C_coo = C.tocoo()
        ii, jj, cij = C_coo.row, C_coo.col, C_coo.data
        denom = (marg[ii] * marg[jj])
        valid = (cij > 0) & (denom > 0)

        ii = ii[valid]; jj = jj[valid]; cij = cij[valid]; denom = denom[valid]

        pmi = np.log((cij * total) / denom)
        ppmi = np.maximum(pmi, 0.0)

        P = coo_matrix((ppmi, (ii, jj)), shape=(n, n)).tocsr()
        P.sum_duplicates()

        row_norms = np.sqrt(P.multiply(P).sum(axis=1)).A1
        row_norms[row_norms == 0] = 1e-12

        self.vocab_ppmi = vocab
        self.idx_ppmi = idx
        self.C_counts = C
        self.marg_counts = marg
        self.total_mass = total
        self.P_ppmi = P
        self.ppmi_norms = row_norms

        if verbose:
            print("\nPPMI metric tables built.")
            print("Vocab size:", len(vocab))
            print("Nonzeros in PPMI matrix:", P.nnz)

    # ---------- PairScore parts ----------
    def _ppmi_cosine(self, i_token: str, j_token: str) -> float:
        idx = self.idx_ppmi
        if i_token not in idx or j_token not in idx:
            return 0.0
        i = idx[i_token]; j = idx[j_token]
        dot = self.P_ppmi[i].multiply(self.P_ppmi[j]).sum()
        return float(dot / (self.ppmi_norms[i] * self.ppmi_norms[j]))

    def _co_use(self, i_token: str, j_token: str) -> float:
        idx = self.idx_ppmi
        if i_token not in idx or j_token not in idx:
            return 0.0
        i = idx[i_token]; j = idx[j_token]
        cij = float(self.C_counts[i, j])
        if cij <= 0:
            return 0.0
        mi = float(self.marg_counts[i])
        mj = float(self.marg_counts[j])
        if mi <= 0 or mj <= 0:
            return 0.0
        return float(max(cij / mi, cij / mj))

    def pair_score(self, i_token: str, j_token: str) -> float:
        sim = self._ppmi_cosine(i_token, j_token)
        penalty = 1.0 - self._co_use(i_token, j_token)
        penalty = max(0.0, min(1.0, penalty))
        return float(sim * penalty)

    # ---------- Eval set ----------
    def _build_eval_set(self):
        rng = np.random.default_rng(self.seed)

        order = np.argsort(-self.marg_counts)
        tokens_sorted = self.vocab_ppmi[order].to_numpy()

        cutoff = int(len(tokens_sorted) * self.keep_top_frac)
        tokens_70 = tokens_sorted[:cutoff]

        n = len(tokens_70)
        head_end = max(1, int(0.05 * n))
        mid_end  = max(head_end + 1, int(0.30 * n))

        head = tokens_70[:head_end]
        mid  = tokens_70[head_end:mid_end]
        tail = tokens_70[mid_end:]

        def sample(arr, k):
            if len(arr) <= k:
                return list(arr)
            return list(rng.choice(arr, size=k, replace=False))

        eval_tokens = sample(head, 100) + sample(mid, 100) + sample(tail, 100)
        eval_tokens = list(dict.fromkeys(eval_tokens))  # de-dup preserving order

        # Top-up
        if len(eval_tokens) < self.n_eval:
            pool = list(tokens_70)
            rng.shuffle(pool)
            have = set(eval_tokens)
            for t in pool:
                if t not in have:
                    eval_tokens.append(t)
                    have.add(t)
                if len(eval_tokens) >= self.n_eval:
                    break

        eval_tokens = eval_tokens[:self.n_eval]
        rng.shuffle(eval_tokens)
        return eval_tokens

    def _load_or_build_eval_set(self, verbose=True):
        path = self.eval_path
        if path.exists():
            df = pd.read_csv(path)
            tokens = df["ingredient"].astype(str).tolist()
            if len(tokens) != self.n_eval:
                tokens = self._build_eval_set()
                pd.DataFrame({"ingredient": tokens}).to_csv(path, index=False)
                if verbose:
                    print(f"\nLoaded eval set but it was not {self.n_eval}; rebuilt & re-saved:", path, "| n =", len(tokens))
            else:
                if verbose:
                    print("\nLoaded frozen eval set:", path, "| n =", len(tokens))
            return tokens

        tokens = self._build_eval_set()
        pd.DataFrame({"ingredient": tokens}).to_csv(path, index=False)
        if verbose:
            print("\nSaved frozen eval set:", path, "| n =", len(tokens))
        return tokens

    # ---------- Rank weighting ----------
    @staticmethod
    def rank_weights(k: int, scheme: str = "log") -> np.ndarray:
        r = np.arange(1, k + 1, dtype=float)
        if scheme == "log":
            w = 1.0 / np.log2(r + 1.0)
        elif scheme == "reciprocal":
            w = 1.0 / r
        else:
            raise ValueError("scheme must be 'log' or 'reciprocal'")
        return w / w.sum()

    def _aggregate_ranked(self, scores, scheme: str = "log") -> float:
        scores = np.asarray(scores, dtype=float)
        if scores.size == 0:
            return 0.0
        w = self.rank_weights(len(scores), scheme=scheme)
        return float(np.sum(w * scores))

    # ---------- Main scoring function ----------
    def score(
        self,
        kv,
        topk: int = 10,
        rank_scheme: str = "log",
        exclude_prefixes=("recipe::",),
        min_candidate_marg: float = 0.0,
        eval_tokens=None,
    ):
        if self.idx_ppmi is None or self.P_ppmi is None or self.eval_ingredients is None:
            raise RuntimeError("Metric not built. Call metric.build() once first.")

        if eval_tokens is None:
            eval_tokens = self.eval_ingredients

        model_vocab = set(kv.key_to_index.keys())

        # optional frequency filter lookup (from PPMI marginals)
        marg_lookup = {t: float(self.marg_counts[self.idx_ppmi[t]]) for t in model_vocab if t in self.idx_ppmi}

        per_query_scores = []
        missing_queries = 0
        empty_lists = 0

        for q in eval_tokens:
            if q not in model_vocab or q not in self.idx_ppmi:
                missing_queries += 1
                continue

            raw = kv.most_similar(q, topn=topk * 5)

            cands = []
            for cand, _sim in raw:
                if cand == q:
                    continue
                if any(str(cand).startswith(p) for p in exclude_prefixes):
                    continue
                if cand not in self.idx_ppmi:
                    continue
                if min_candidate_marg > 0 and marg_lookup.get(cand, 0.0) < min_candidate_marg:
                    continue
                cands.append(cand)
                if len(cands) >= topk:
                    break

            if not cands:
                empty_lists += 1
                continue

            scores = [self.pair_score(q, c) for c in cands]  # in rank order
            per_query_scores.append(self._aggregate_ranked(scores, scheme=rank_scheme))

        overall = float(np.mean(per_query_scores)) if per_query_scores else 0.0
        meta = {
            "eval_set_size": len(eval_tokens),
            "scored_queries": len(per_query_scores),
            "missing_queries": missing_queries,
            "empty_lists": empty_lists,
            "topk": topk,
            "rank_scheme": rank_scheme,
        }
        return overall, per_query_scores, meta


In [3]:
# ===========================================================
# NEXT CELL — Iteration 7 training + metric scoring + sanity check
# (ONLY change vs Iter 4: num_walks 20 → 40)
# Uses MORE WORKERS for speed (Mac M4 / 24GB RAM)
# ===========================================================

import networkx as nx
from node2vec import Node2Vec
from pathlib import Path
import numpy as np

# -----------------------------------------------------------
# 0) Ensure metric object exists & is built (build tables only once)
# -----------------------------------------------------------
EDGES_PATH = Path("../data/processed/ingredient_cooccurrence_edges.csv")
EVAL_PATH  = Path("../data/processed/eval_ingredients_300_seed42.csv")

if "metric" not in globals():
    metric = CoocPPMIRankMetric(edges_path=EDGES_PATH, eval_path=EVAL_PATH, seed=42, n_eval=300)

if metric.eval_ingredients is None:
    metric.build(verbose=True)

# -----------------------------------------------------------
# 1) Build the co-occurrence graph G (reuse edges already loaded in metric)
# -----------------------------------------------------------
edges_df = metric.edges_df  # already loaded during metric.build()

G = nx.Graph()
G.add_weighted_edges_from(
    edges_df[["ingredient_a", "ingredient_b", "weight"]].itertuples(index=False, name=None),
    weight="weight",
)

print("\nGraph rebuilt for Node2Vec training.")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# -----------------------------------------------------------
# 2) Iteration 7 hyperparameters
# -----------------------------------------------------------
ITER = 7
DIM = 128
WALK_LEN = 60
NUM_WALKS = 40      # <-- ONLY change vs Iteration 4
P = 1
Q = 4
WINDOW = 5
SEED = 42

# -----------------------------------------------------------
# 2b) Workers (increase for speed)
# - Node2Vec(workers=...) controls random-walk generation
# - fit(workers=...) controls Word2Vec (skip-gram) training
#
# Safe starting point on your machine: 8
# If you see instability or overheating, drop to 6.
# -----------------------------------------------------------
WALK_WORKERS = 8
W2V_WORKERS  = 8

# -----------------------------------------------------------
# 3) Train Node2Vec (UNWEIGHTED walks)
# -----------------------------------------------------------
node2vec_iter7 = Node2Vec(
    G,
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    # IMPORTANT: unweighted walks -> do NOT pass weight_key
    workers=WALK_WORKERS,
    seed=SEED,
)

model_iter7 = node2vec_iter7.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=W2V_WORKERS,
)

print(
    f"\nIteration {ITER} training complete "
    f"(UNWEIGHTED, p={P}, q={Q}, window={WINDOW}, num_walks={NUM_WALKS}, "
    f"walk_workers={WALK_WORKERS}, w2v_workers={W2V_WORKERS})."
)

# -----------------------------------------------------------
# 4) Save embeddings (keep filename explicit for traceability)
# -----------------------------------------------------------
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter7 = OUT_DIR / (
    f"ingredient_node2vec_iter{ITER}_UNWEIGHTED_p{P}_q{Q}_w{WINDOW}_r{NUM_WALKS}.txt"
)

model_iter7.wv.save_word2vec_format(str(emb_path_iter7))

print("Saved embeddings to:", emb_path_iter7)
print("Vectors:", len(model_iter7.wv), "| dim:", model_iter7.wv.vector_size)

# -----------------------------------------------------------
# 5) Run the metric on Iteration 7 (rank-aware)
# -----------------------------------------------------------
TOPK = 10
RANK_SCHEME = "log"  # keep consistent across iterations for fair comparison

score_iter7, per_q_iter7, meta_iter7 = metric.score(
    model_iter7.wv,
    topk=TOPK,
    rank_scheme=RANK_SCHEME,
    exclude_prefixes=("recipe::",),
    min_candidate_marg=0.0,
)

print("\n" + "=" * 70)
print(f"METRIC RESULT — Co-occurrence Node2Vec Iteration {ITER} (UNWEIGHTED) | rank={RANK_SCHEME}")
print(f"Eval set size: {meta_iter7['eval_set_size']}")
print(f"Scored queries: {meta_iter7['scored_queries']}")
print(f"Missing queries: {meta_iter7['missing_queries']}")
print(f"Empty candidate lists: {meta_iter7['empty_lists']}")
print(f"Top-k: {meta_iter7['topk']}")
print(f"ModelScore: {score_iter7:.6f}")
print("=" * 70)

if per_q_iter7:
    arr = np.array(per_q_iter7)
    print("\nPer-query Metric@k summary:")
    for p in [0, 10, 25, 50, 75, 90, 100]:
        print(f"  p{p:>3}: {np.percentile(arr, p):.6f}")

# -----------------------------------------------------------
# 6) Visual sanity check — Top-10 neighbours for a few probes
# -----------------------------------------------------------
def print_topk(kv, probes, topn=10):
    vocab = set(kv.key_to_index.keys())
    for p in probes:
        if p not in vocab:
            print(f"\nProbe missing: {p}")
            continue
        print("\n" + "=" * 60)
        print("Probe:", p)
        for i, (nbr, sim) in enumerate(kv.most_similar(p, topn=topn), 1):
            print(f"{i:02d}. {nbr:<30} cosine={sim:.3f}")

probes_to_check = ["real butter", "pure olive oil", "milk", "flour", "chicken"]
print_topk(model_iter7.wv, probes_to_check, topn=10)


Loaded edges_df for metric reference.
Rows: 419913
Columns: ['ingredient_a', 'ingredient_b', 'weight']

PPMI metric tables built.
Vocab size: 2698
Nonzeros in PPMI matrix: 839826

Loaded frozen eval set: ../data/processed/eval_ingredients_300_seed42.csv | n = 300

Graph rebuilt for Node2Vec training.
Nodes: 2698
Edges: 419913


Generating walks (CPU: 8): 100%|██████████| 5/5 [00:39<00:00,  7.89s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_


Iteration 7 training complete (UNWEIGHTED, p=1, q=4, window=5, num_walks=40, walk_workers=8, w2v_workers=8).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter7_UNWEIGHTED_p1_q4_w5_r40.txt
Vectors: 2698 | dim: 128

METRIC RESULT — Co-occurrence Node2Vec Iteration 7 (UNWEIGHTED) | rank=log
Eval set size: 300
Scored queries: 300
Missing queries: 0
Empty candidate lists: 0
Top-k: 10
ModelScore: 0.295424

Per-query Metric@k summary:
  p  0: 0.109447
  p 10: 0.177580
  p 25: 0.234492
  p 50: 0.295980
  p 75: 0.354377
  p 90: 0.407364
  p100: 0.510183

Probe: real butter
01. colored crystal sugar          cosine=0.556
02. english toffee bit             cosine=0.537
03. chocolate wafer                cosine=0.534
04. butter flavoring               cosine=0.517
05. anise extract                  cosine=0.514
06. crisco shortening              cosine=0.514
07. milk chocolate kisse           cosine=0.514
08. mint chocolate                 cosine=0.513
09. semisweet choco

In [4]:
# ===========================================================
# NEXT CELL — Iteration 8 training + metric scoring + sanity check
# (ONLY change vs Iter 7: window 5 → 10)
# ===========================================================

import networkx as nx
from node2vec import Node2Vec
from pathlib import Path
import numpy as np

# -----------------------------------------------------------
# 0) Ensure metric object exists & is built (build tables only once)
# -----------------------------------------------------------
EDGES_PATH = Path("../data/processed/ingredient_cooccurrence_edges.csv")
EVAL_PATH  = Path("../data/processed/eval_ingredients_300_seed42.csv")

if "metric" not in globals():
    metric = CoocPPMIRankMetric(edges_path=EDGES_PATH, eval_path=EVAL_PATH, seed=42, n_eval=300)

if metric.eval_ingredients is None:
    metric.build(verbose=True)

# -----------------------------------------------------------
# 1) Build the co-occurrence graph G (reuse edges already loaded in metric)
# -----------------------------------------------------------
edges_df = metric.edges_df  # already loaded during metric.build()

G = nx.Graph()
G.add_weighted_edges_from(
    edges_df[["ingredient_a", "ingredient_b", "weight"]].itertuples(index=False, name=None),
    weight="weight",
)

print("\nGraph rebuilt for Node2Vec training.")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# -----------------------------------------------------------
# 2) Iteration 8 hyperparameters
# -----------------------------------------------------------
ITER = 8
DIM = 128
WALK_LEN = 60
NUM_WALKS = 40      # same as Iter 7
P = 1
Q = 4
WINDOW = 10         # <-- ONLY change vs Iteration 7 (was 5)
SEED = 42

# -----------------------------------------------------------
# 2b) Workers (keep same as Iter 7)
# -----------------------------------------------------------
WALK_WORKERS = 8
W2V_WORKERS  = 8

# -----------------------------------------------------------
# 3) Train Node2Vec (UNWEIGHTED walks)
# -----------------------------------------------------------
node2vec_iter8 = Node2Vec(
    G,
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    # IMPORTANT: unweighted walks -> do NOT pass weight_key
    workers=WALK_WORKERS,
    seed=SEED,
)

model_iter8 = node2vec_iter8.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=W2V_WORKERS,
)

print(
    f"\nIteration {ITER} training complete "
    f"(UNWEIGHTED, p={P}, q={Q}, window={WINDOW}, num_walks={NUM_WALKS}, "
    f"walk_workers={WALK_WORKERS}, w2v_workers={W2V_WORKERS})."
)

# -----------------------------------------------------------
# 4) Save embeddings (explicit filename for traceability)
# -----------------------------------------------------------
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter8 = OUT_DIR / (
    f"ingredient_node2vec_iter{ITER}_UNWEIGHTED_p{P}_q{Q}_w{WINDOW}_r{NUM_WALKS}.txt"
)

model_iter8.wv.save_word2vec_format(str(emb_path_iter8))

print("Saved embeddings to:", emb_path_iter8)
print("Vectors:", len(model_iter8.wv), "| dim:", model_iter8.wv.vector_size)

# -----------------------------------------------------------
# 5) Run the metric on Iteration 8 (rank-aware)
# -----------------------------------------------------------
TOPK = 10
RANK_SCHEME = "log"  # keep consistent across iterations

score_iter8, per_q_iter8, meta_iter8 = metric.score(
    model_iter8.wv,
    topk=TOPK,
    rank_scheme=RANK_SCHEME,
    exclude_prefixes=("recipe::",),
    min_candidate_marg=0.0,
)

print("\n" + "=" * 70)
print(f"METRIC RESULT — Co-occurrence Node2Vec Iteration {ITER} (UNWEIGHTED) | rank={RANK_SCHEME}")
print(f"Eval set size: {meta_iter8['eval_set_size']}")
print(f"Scored queries: {meta_iter8['scored_queries']}")
print(f"Missing queries: {meta_iter8['missing_queries']}")
print(f"Empty candidate lists: {meta_iter8['empty_lists']}")
print(f"Top-k: {meta_iter8['topk']}")
print(f"ModelScore: {score_iter8:.6f}")
print("=" * 70)

if per_q_iter8:
    arr = np.array(per_q_iter8)
    print("\nPer-query Metric@k summary:")
    for p in [0, 10, 25, 50, 75, 90, 100]:
        print(f"  p{p:>3}: {np.percentile(arr, p):.6f}")

# -----------------------------------------------------------
# 6) Visual sanity check — Top-10 neighbours for a few probes
# -----------------------------------------------------------
def print_topk(kv, probes, topn=10):
    vocab = set(kv.key_to_index.keys())
    for p in probes:
        if p not in vocab:
            print(f"\nProbe missing: {p}")
            continue
        print("\n" + "=" * 60)
        print("Probe:", p)
        for i, (nbr, sim) in enumerate(kv.most_similar(p, topn=topn), 1):
            print(f"{i:02d}. {nbr:<30} cosine={sim:.3f}")

probes_to_check = ["real butter", "pure olive oil", "milk", "flour", "chicken"]
print_topk(model_iter8.wv, probes_to_check, topn=10)



Graph rebuilt for Node2Vec training.
Nodes: 2698
Edges: 419913


Generating walks (CPU: 8): 100%|██████████| 5/5 [00:45<00:00,  9.02s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_


Iteration 8 training complete (UNWEIGHTED, p=1, q=4, window=10, num_walks=40, walk_workers=8, w2v_workers=8).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter8_UNWEIGHTED_p1_q4_w10_r40.txt
Vectors: 2698 | dim: 128

METRIC RESULT — Co-occurrence Node2Vec Iteration 8 (UNWEIGHTED) | rank=log
Eval set size: 300
Scored queries: 300
Missing queries: 0
Empty candidate lists: 0
Top-k: 10
ModelScore: 0.340060

Per-query Metric@k summary:
  p  0: 0.109120
  p 10: 0.184872
  p 25: 0.261385
  p 50: 0.346187
  p 75: 0.420593
  p 90: 0.480811
  p100: 0.617457

Probe: real butter
01. persimmon pulp                 cosine=0.630
02. flour                          cosine=0.593
03. dutch cocoa                    cosine=0.588
04. crisco                         cosine=0.582
05. egg                            cosine=0.582
06. semisweet chocolate chunk      cosine=0.579
07. shortening                     cosine=0.575
08. decorator sugar                cosine=0.571
09. soda         

In [5]:
# ===========================================================
# DIAGNOSTIC CELL — Does the metric actually help vs cosine?
# What this does:
# - For a few probe ingredients, it compares:
#   (A) Node2Vec cosine top-N
#   (B) The same candidates re-ranked by your metric PairScore
# - Shows how much the metric reorders the list, and whether
#   the “weird” cosine neighbours get penalized.
#
# Interpretation:
# - If PairScore-ranked lists look MORE like substitutes (or at least more coherent)
#   than cosine lists, your metric is doing useful work.
# - If PairScore-ranked lists look basically the same OR equally weird,
#   that's a signal you may need to adjust the metric (penalty strength, etc.).
# ===========================================================

import numpy as np

def diagnostic_metric_vs_cosine(
    metric,
    kv,
    probes,
    topk=10,
    candidate_pool=200,
    rank_scheme="log",     # not used for rerank, but keeps the output consistent with your setup
    exclude_prefixes=("recipe::",),
):
    """
    Prints, per probe:
    1) Cosine top-k neighbours
    2) Metric PairScore re-ranked top-k neighbours (from cosine top candidate_pool)
    3) Reordering strength + penalty visibility
    """
    model_vocab = set(kv.key_to_index.keys())
    idx_ppmi = metric.idx_ppmi

    def ok_token(t):
        if any(str(t).startswith(p) for p in exclude_prefixes):
            return False
        return True

    for probe in probes:
        if probe not in model_vocab:
            print("\n" + "=" * 80)
            print(f"Probe missing in embeddings vocab: {probe}")
            continue
        if probe not in idx_ppmi:
            print("\n" + "=" * 80)
            print(f"Probe missing in PPMI vocab (metric reference): {probe}")
            continue

        # 1) Candidate pool by cosine
        raw = kv.most_similar(probe, topn=max(candidate_pool, topk * 5))
        raw = [(c, s) for (c, s) in raw if c != probe and ok_token(c) and c in idx_ppmi]

        if not raw:
            print("\n" + "=" * 80)
            print(f"Probe has no usable candidates after filtering: {probe}")
            continue

        # keep only pool size
        pool = raw[:candidate_pool]

        # 2) Compute metric components for pool and re-rank by PairScore
        rows = []
        for rank, (cand, cos) in enumerate(pool, start=1):
            sim_ctx = metric._ppmi_cosine(probe, cand)
            co_use = metric._co_use(probe, cand)
            ps = sim_ctx * max(0.0, min(1.0, (1.0 - co_use)))
            rows.append((cand, cos, sim_ctx, co_use, ps, rank))

        # cosine topk
        cosine_top = rows[:topk]

        # pairscore rerank
        reranked = sorted(rows, key=lambda x: x[4], reverse=True)[:topk]

        # 3) Measure reordering: average absolute change in original cosine rank
        # (large means metric is actively changing rankings)
        original_rank_map = {cand: r for (cand, *_rest, r) in rows}
        deltas = [abs(original_rank_map[cand] - (i + 1)) for i, (cand, *_rest) in enumerate(reranked)]
        avg_abs_rank_change = float(np.mean(deltas)) if deltas else 0.0

        # 4) How much co-use penalty is applied in the cosine topk?
        co_use_vals = [x[3] for x in cosine_top]
        avg_co_use_cosine_top = float(np.mean(co_use_vals)) if co_use_vals else 0.0
        max_co_use_cosine_top = float(np.max(co_use_vals)) if co_use_vals else 0.0

        print("\n" + "=" * 80)
        print(f"Probe: {probe}")
        print(f"Candidate pool: {len(pool)} | topk={topk}")
        print(f"Avg |rank change| in PairScore topk vs cosine ordering: {avg_abs_rank_change:.2f}")
        print(f"Cosine-topk co-use: mean={avg_co_use_cosine_top:.4f}, max={max_co_use_cosine_top:.4f}")

        print("\n(A) Cosine top-k (with metric components shown):")
        for i, (cand, cos, sim_ctx, co_use, ps, cos_rank) in enumerate(cosine_top, start=1):
            print(
                f"{i:02d}. {cand:<28} "
                f"cos={cos:.3f}  ctx={sim_ctx:.3f}  co_use={co_use:.4f}  PairScore={ps:.3f}"
            )

        print("\n(B) PairScore re-ranked top-k (from cosine candidate pool):")
        for i, (cand, cos, sim_ctx, co_use, ps, cos_rank) in enumerate(reranked, start=1):
            print(
                f"{i:02d}. {cand:<28} "
                f"PairScore={ps:.3f}  ctx={sim_ctx:.3f}  co_use={co_use:.4f}  cos={cos:.3f}  (was cosine-rank {cos_rank})"
            )

        # 5) Simple “verdict” hints (doesn't change anything, just guidance)
        # - If reranked list is almost identical + rank change tiny: penalty isn't influencing much.
        # - If reranked list is very different but still weird: metric may not align with substitution.
        overlap = len(set([x[0] for x in cosine_top]).intersection([x[0] for x in reranked]))
        print(f"\nOverlap between cosine-topk and PairScore-topk: {overlap}/{topk}")
        if avg_abs_rank_change < 1.0:
            print("Hint: PairScore barely reorders cosine list → co-use penalty may be too weak OR candidates already low co-use.")
        elif avg_abs_rank_change > (topk / 2):
            print("Hint: PairScore strongly reorders candidates → metric is actively doing work (now inspect if reranked looks more substitute-like).")


# -----------------------------------------------------------
# RUN THE DIAGNOSTIC
# -----------------------------------------------------------
# Use the latest model you trained (e.g., model_iter8.wv) or any kv
# Example: kv_current = model_iter8.wv  (or model_iter7.wv)
kv_current = model_iter7.wv  # <-- change to whichever iteration you want to inspect

probes_to_debug = [
    "real butter",
    "pure olive oil",
    "milk",
    "flour",
    "chicken",
]

diagnostic_metric_vs_cosine(
    metric=metric,
    kv=kv_current,
    probes=probes_to_debug,
    topk=10,
    candidate_pool=200,   # increase to 500 if you want deeper reranking
    exclude_prefixes=("recipe::",),
)



Probe: real butter
Candidate pool: 200 | topk=10
Avg |rank change| in PairScore topk vs cosine ordering: 105.10
Cosine-topk co-use: mean=0.0000, max=0.0000

(A) Cosine top-k (with metric components shown):
01. colored crystal sugar        cos=0.556  ctx=0.158  co_use=0.0000  PairScore=0.158
02. english toffee bit           cos=0.537  ctx=0.116  co_use=0.0000  PairScore=0.116
03. chocolate wafer              cos=0.534  ctx=0.116  co_use=0.0000  PairScore=0.116
04. butter flavoring             cos=0.517  ctx=0.101  co_use=0.0000  PairScore=0.101
05. anise extract                cos=0.514  ctx=0.104  co_use=0.0000  PairScore=0.104
06. crisco shortening            cos=0.514  ctx=0.117  co_use=0.0000  PairScore=0.117
07. milk chocolate kisse         cos=0.514  ctx=0.106  co_use=0.0000  PairScore=0.106
08. mint chocolate               cos=0.513  ctx=0.084  co_use=0.0000  PairScore=0.084
09. semisweet chocolate          cos=0.511  ctx=0.161  co_use=0.0000  PairScore=0.161
10. chocolate cooki

In [6]:
# ===========================================================
# EXTRA VALIDATION CELL (add at end):
# "Known substitutes" sanity-check + correlation with Metric@k
#
# What this does:
# - Uses a small, fixed substitute dictionary (weak supervision)
# - For each probe ingredient, checks if any "known substitutes" appear
#   in the model's top-k list
# - Computes: Hits@K, MRR@K, and per-probe diagnostics
#
# IMPORTANT:
# - This does NOT change your main metric.
# - It is an additional alignment check to confirm that higher metric
#   tends to rank plausible substitutes higher.
# ===========================================================

import numpy as np
import pandas as pd

# ----------------------------
# (0) Curated substitute map
# ----------------------------
SUBS = {
    # Fats / oils
    "butter": ["margarine", "shortening", "ghee", "unsalted margarine", "vegan butter"],
    "real butter": ["margarine", "shortening", "ghee", "unsalted margarine", "vegan butter"],
    "olive oil": ["vegetable oil", "canola oil", "sunflower oil", "grapeseed oil", "avocado oil"],
    "pure olive oil": ["vegetable oil", "canola oil", "sunflower oil", "grapeseed oil", "avocado oil"],
    "vegetable oil": ["canola oil", "sunflower oil", "corn oil", "safflower oil"],

    # Dairy / liquids
    "milk": ["soy milk", "almond milk", "oat milk", "evaporated milk", "coconut milk"],
    "cream": ["heavy cream", "whipping cream", "half-and-half", "coconut cream"],
    "buttermilk": ["milk", "yogurt", "milk + lemon juice", "milk + vinegar"],  # last two are conceptual

    # Flour / starch
    "all-purpose flour": ["bread flour", "cake flour", "whole wheat flour", "gluten-free flour"],
    "flour": ["all-purpose flour", "bread flour", "cake flour", "whole wheat flour", "gluten-free flour"],
    "cornstarch": ["arrowroot", "potato starch", "tapioca starch"],
    "breadcrumbs": ["panko", "crushed crackers", "oat crumbs"],

    # Eggs / binders
    "egg": ["flax egg", "chia egg", "applesauce", "silken tofu"],

    # Proteins (contextual)
    "chicken": ["turkey", "tofu", "tempeh", "seitan"],
    "ground beef": ["ground turkey", "ground chicken", "lentils"],

    # Sweeteners
    "sugar": ["brown sugar", "honey", "maple syrup", "agave nectar"],
    "brown sugar": ["sugar", "coconut sugar"],

    # Acids
    "lemon juice": ["lime juice", "vinegar", "apple cider vinegar"],
    "vinegar": ["lemon juice", "white wine vinegar", "rice vinegar"],
}

# ----------------------------
# (1) Helper: normalize tokens
# ----------------------------
def norm_tok(s: str) -> str:
    return str(s).strip().lower()

# ----------------------------
# (2) Evaluate model top-k for substitute hits
# ----------------------------
def eval_substitute_hits(kv, subs_map, k_list=(10, 20, 50), verbose=True):
    vocab = set(kv.key_to_index.keys())

    rows = []
    for probe_raw, subs_raw in subs_map.items():
        probe = norm_tok(probe_raw)
        subs = [norm_tok(x) for x in subs_raw]

        in_vocab_probe = probe in vocab
        subs_in_vocab = [s for s in subs if s in vocab]

        if not in_vocab_probe:
            rows.append({
                "probe": probe_raw,
                "probe_in_vocab": False,
                "n_subs_in_vocab": len(subs_in_vocab),
                "topk_list": None,
                "best_rank": None,
                "best_sub": None,
                **{f"hit@{k}": np.nan for k in k_list},
                **{f"mrr@{k}": np.nan for k in k_list},
            })
            continue

        # Get a reasonably large ranked list once
        max_k = max(k_list)
        ranked = [cand for cand, _ in kv.most_similar(probe, topn=max_k * 3) if cand != probe]

        # compute best rank among known substitutes
        best_rank = None
        best_sub = None
        for r, cand in enumerate(ranked, start=1):
            if cand in subs_in_vocab:
                best_rank = r
                best_sub = cand
                break

        out = {
            "probe": probe_raw,
            "probe_in_vocab": True,
            "n_subs_in_vocab": len(subs_in_vocab),
            "best_rank": best_rank,
            "best_sub": best_sub,
        }

        for k in k_list:
            topk = ranked[:k]
            hit = 1 if any(c in subs_in_vocab for c in topk) else 0
            mrr = (1.0 / best_rank) if (best_rank is not None and best_rank <= k) else 0.0
            out[f"hit@{k}"] = hit
            out[f"mrr@{k}"] = mrr

        rows.append(out)

        if verbose:
            # show quick diagnostics
            print("\n" + "=" * 70)
            print(f"Probe: {probe_raw} | subs_in_vocab={len(subs_in_vocab)}")
            if subs_in_vocab:
                print("Known subs (in vocab):", subs_in_vocab[:12], ("..." if len(subs_in_vocab) > 12 else ""))
            else:
                print("Known subs (in vocab): NONE (your vocab may not contain these exact tokens)")
            print("Top-10:", ranked[:10])
            if best_rank is None:
                print("Best known-sub rank: NOT FOUND in searched list")
            else:
                print(f"Best known-sub rank: {best_rank} (sub='{best_sub}')")

    df = pd.DataFrame(rows)

    summary = {}
    for k in k_list:
        # only consider probes that exist in vocab AND have at least 1 substitute in vocab
        mask = (df["probe_in_vocab"] == True) & (df["n_subs_in_vocab"] > 0)
        denom = int(mask.sum())
        if denom == 0:
            summary[f"Hits@{k}"] = np.nan
            summary[f"MRR@{k}"] = np.nan
        else:
            summary[f"Hits@{k}"] = float(df.loc[mask, f"hit@{k}"].mean())
            summary[f"MRR@{k}"] = float(df.loc[mask, f"mrr@{k}"].mean())

    summary["n_probes_total"] = len(df)
    summary["n_probes_usable"] = int(((df["probe_in_vocab"] == True) & (df["n_subs_in_vocab"] > 0)).sum())

    return df, summary

# ----------------------------
# (3) Run on your latest model in memory
# ----------------------------
# Assumes you just trained something like model_iter8 / model_iter7 and ran metric.score(...)
# Choose which one you want to validate:
if "model_iter8" in globals():
    kv_to_check = model_iter8.wv
    iter_name = "iter8"
elif "model_iter7" in globals():
    kv_to_check = model_iter7.wv
    iter_name = "iter7"
elif "model_iter4" in globals():
    kv_to_check = model_iter4.wv
    iter_name = "iter4"
else:
    raise RuntimeError("No model_iter8/model_iter7/model_iter4 found in memory to validate.")

print("\n" + "#" * 80)
print(f"KNOWN-SUBSTITUTE VALIDATION for {iter_name}")
print("#" * 80)

subs_df, subs_summary = eval_substitute_hits(
    kv_to_check,
    SUBS,
    k_list=(10, 20, 50),
    verbose=True
)

print("\n" + "=" * 70)
print("SUMMARY (only probes with >=1 known-sub token present in your vocab)")
for k, v in subs_summary.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")
print("=" * 70)

# Optional: display as a table sorted by best_rank (easiest wins at top)
display(subs_df.sort_values(by=["probe_in_vocab", "n_subs_in_vocab", "best_rank"], ascending=[False, False, True]))



################################################################################
KNOWN-SUBSTITUTE VALIDATION for iter8
################################################################################

Probe: real butter | subs_in_vocab=4
Known subs (in vocab): ['margarine', 'shortening', 'ghee', 'vegan butter'] 
Top-10: ['persimmon pulp', 'flour', 'dutch cocoa', 'crisco', 'egg', 'semisweet chocolate chunk', 'shortening', 'decorator sugar', 'soda', 'cake-and-pastry flour']
Best known-sub rank: 7 (sub='shortening')

Probe: pure olive oil | subs_in_vocab=3
Known subs (in vocab): ['canola oil', 'sunflower oil', 'grapeseed oil'] 
Top-10: ['veal chop', 'anchovy packed in oil', 'spanish chorizo', 'parsley', 'basil', 'trout', 'bass', 'orange roughy', 'oregano', 'lemon thyme']
Best known-sub rank: NOT FOUND in searched list

Probe: milk | subs_in_vocab=2
Known subs (in vocab): ['almond milk', 'coconut milk'] 
Top-10: ['flour', 'egg', 'margarine', 'cream', 'cream cheese', 'nutmeg', 'wooden stic

,probe,probe_in_vocab,n_subs_in_vocab,topk_list,best_rank,best_sub,hit@10,hit@20,hit@50,mrr@10,mrr@20,mrr@50
1,real butter,True,4,NaN,7.0,shortening,1.0,1.0,1.0,0.142857,0.142857,0.142857
13,chicken,True,4,NaN,20.0,turkey,0.0,1.0,1.0,0.000000,0.050000,0.050000
6,cream,True,3,NaN,2.0,whipping cream,1.0,1.0,1.0,0.500000,0.500000,0.500000
17,lemon juice,True,3,NaN,55.0,apple cider vinegar,0.0,0.0,0.0,0.000000,0.000000,0.000000
9,flour,True,3,NaN,75.0,cake flour,0.0,0.0,0.0,0.000000,0.000000,0.000000
3,pure olive oil,True,3,NaN,NaN,None,0.0,0.0,0.0,0.000000,0.000000,0.000000
10,cornstarch,True,3,NaN,NaN,None,0.0,0.0,0.0,0.000000,0.000000,0.000000
7,buttermilk,True,2,NaN,6.0,milk,1.0,1.0,1.0,0.166667,0.166667,0.166667
18,vinegar,True,2,NaN,12.0,lemon juice,0.0,1.0,1.0,0.000000,0.083333,0.083333
12,egg,True,2,NaN,43.0,applesauce,0.0,0.0,1.0,0.000000,0.000000,0.023256


In [7]:
# ===========================================================
# EXTRA VALIDATION — Known-Substitute Score + Combined Metric
# Adds a substitution-focused check to complement the PPMI metric.
# ===========================================================

import numpy as np
import pandas as pd

# ----------------------------
# (A) Define a small substitute dictionary (edit/expand over time)
# IMPORTANT: tokens must match your vocab exactly.
# ----------------------------
KNOWN_SUBS = {
    "real butter": ["margarine", "shortening", "ghee", "vegan butter"],
    "pure olive oil": ["canola oil", "sunflower oil", "grapeseed oil"],
    "milk": ["almond milk", "coconut milk", "oat milk", "soy milk"],
    "cream": ["whipping cream", "half-and-half", "coconut cream"],
    "buttermilk": ["milk", "yogurt"],
    "flour": ["bread flour", "cake flour", "gluten-free flour"],
    "cornstarch": ["arrowroot", "potato starch", "tapioca starch"],
    "egg": ["applesauce", "silken tofu", "flax egg"],
    "chicken": ["turkey", "tofu", "tempeh", "seitan"],
    "lemon juice": ["lime juice", "vinegar", "apple cider vinegar"],
    "vinegar": ["lemon juice", "rice vinegar"],
}

# ----------------------------
# (B) Known-sub evaluation
# ----------------------------
def known_substitute_score(kv, known_subs: dict, topk=10, search_k=300):
    """
    Computes Hits@K and MRR@K for a curated known-substitute dictionary.
    - search_k: how far into most_similar to search (>= topk). Use ~300 for stability.
    Returns: (summary_dict, per_probe_df)
    """
    vocab = set(kv.key_to_index.keys())

    rows = []
    hits_k = []
    mrr_k = []

    for probe, subs in known_subs.items():
        probe_in_vocab = probe in vocab
        subs_in_vocab = [s for s in subs if s in vocab]

        if not probe_in_vocab or len(subs_in_vocab) == 0:
            rows.append({
                "probe": probe,
                "probe_in_vocab": probe_in_vocab,
                "n_subs_in_vocab": len(subs_in_vocab),
                "best_rank": np.nan,
                "best_sub": None,
                f"hit@{topk}": np.nan,
                f"mrr@{topk}": np.nan,
            })
            continue

        # get a long list then find earliest known substitute
        neigh = [w for (w, _) in kv.most_similar(probe, topn=search_k)]

        best_rank = None
        best_sub = None
        for r, w in enumerate(neigh, start=1):
            if w in subs_in_vocab:
                best_rank = r
                best_sub = w
                break

        hit = 1.0 if (best_rank is not None and best_rank <= topk) else 0.0
        rr  = (1.0 / best_rank) if (best_rank is not None and best_rank <= topk) else 0.0

        hits_k.append(hit)
        mrr_k.append(rr)

        rows.append({
            "probe": probe,
            "probe_in_vocab": True,
            "n_subs_in_vocab": len(subs_in_vocab),
            "best_rank": float(best_rank) if best_rank is not None else np.nan,
            "best_sub": best_sub,
            f"hit@{topk}": hit,
            f"mrr@{topk}": rr,
        })

    per_probe = pd.DataFrame(rows)
    usable = per_probe[(per_probe["probe_in_vocab"] == True) & (per_probe["n_subs_in_vocab"] > 0)]

    summary = {
        "n_probes_total": int(len(per_probe)),
        "n_probes_usable": int(len(usable)),
        f"Hits@{topk}": float(np.mean(hits_k)) if hits_k else 0.0,
        f"MRR@{topk}": float(np.mean(mrr_k)) if mrr_k else 0.0,
    }
    return summary, per_probe


# ----------------------------
# (C) Run your unsupervised metric (already exists) + known-sub metric
# ----------------------------
TOPK = 10
RANK_SCHEME = "log"

# kv to evaluate (change this per iteration)
kv = model_iter8.wv if "model_iter8" in globals() else model_iter7.wv  # fallback example

unsup_score, _, unsup_meta = metric.score(kv, topk=TOPK, rank_scheme=RANK_SCHEME)

sub_summary, sub_table = known_substitute_score(kv, KNOWN_SUBS, topk=TOPK, search_k=300)

# ----------------------------
# (D) Combine them (tunable weights)
# ----------------------------
ALPHA = 0.20  # weight of known-sub score (0.0 -> ignore, 1.0 -> only substitutes)
BETA  = 0.70  # within known-sub: prefer MRR (ranking) over Hits (binary)

sub_score = BETA * sub_summary[f"MRR@{TOPK}"] + (1 - BETA) * sub_summary[f"Hits@{TOPK}"]
final_score = (1 - ALPHA) * unsup_score + ALPHA * sub_score

print("\n" + "=" * 78)
print("COMBINED EVALUATION")
print(f"Unsupervised PPMI/co-use metric@{TOPK} (rank={RANK_SCHEME}): {unsup_score:.6f}")
print(f"Known-substitute Hits@{TOPK}: {sub_summary[f'Hits@{TOPK}']:.6f}  |  MRR@{TOPK}: {sub_summary[f'MRR@{TOPK}']:.6f}")
print(f"Known-sub score (BETA={BETA}): {sub_score:.6f}")
print(f"FINAL combined score (ALPHA={ALPHA}): {final_score:.6f}")
print(f"Probes usable: {sub_summary['n_probes_usable']}/{sub_summary['n_probes_total']}")
print("=" * 78)

# Show the per-probe table sorted by best_rank (good ones at top)
display(sub_table.sort_values(by="best_rank", na_position="last"))



COMBINED EVALUATION
Unsupervised PPMI/co-use metric@10 (rank=log): 0.340060
Known-substitute Hits@10: 0.272727  |  MRR@10: 0.073593
Known-sub score (BETA=0.7): 0.133333
FINAL combined score (ALPHA=0.2): 0.298715
Probes usable: 11/11


,probe,probe_in_vocab,n_subs_in_vocab,best_rank,best_sub,hit@10,mrr@10
3,cream,True,3,2.0,whipping cream,1.0,0.500000
4,buttermilk,True,2,6.0,milk,1.0,0.166667
0,real butter,True,4,7.0,shortening,1.0,0.142857
10,vinegar,True,2,12.0,lemon juice,0.0,0.000000
8,chicken,True,4,20.0,turkey,0.0,0.000000
7,egg,True,2,43.0,applesauce,0.0,0.000000
9,lemon juice,True,3,55.0,apple cider vinegar,0.0,0.000000
5,flour,True,3,75.0,cake flour,0.0,0.000000
6,cornstarch,True,3,184.0,arrowroot,0.0,0.000000
1,pure olive oil,True,3,NaN,None,0.0,0.000000


In [8]:
# ===========================================================
# NEXT BEST ITERATION — Iteration 9
# Goal: maximize metric while staying theory-consistent (DFS-biased Node2Vec)
#
# Rationale (theory-consistent):
# - Keep DFS bias: q=4 (same as your best so far)
# - Keep p=1 (neutral return)
# - Keep unweighted walks (avoid frequency domination)
# - Increase walk_length (60 -> 80) to capture broader structural context
# - Keep num_walks=40 (already helped a lot)
# - Keep window=10 (Iter8 improved a lot; keep it)
# - Increase Word2Vec training epochs (default -> 10) to better fit walks
#
# Outputs:
# - trains Iter9
# - saves embeddings
# - runs metric.score(...)
# - prints percentile summary
# - prints top-10 neighbours for probes
# ===========================================================

import networkx as nx
from node2vec import Node2Vec
from pathlib import Path
import numpy as np

# -----------------------------------------------------------
# 0) Ensure metric object exists & is built (build tables only once)
# -----------------------------------------------------------
EDGES_PATH = Path("../data/processed/ingredient_cooccurrence_edges.csv")
EVAL_PATH  = Path("../data/processed/eval_ingredients_300_seed42.csv")

if "metric" not in globals():
    metric = CoocPPMIRankMetric(edges_path=EDGES_PATH, eval_path=EVAL_PATH, seed=42, n_eval=300)

if metric.eval_ingredients is None:
    metric.build(verbose=True)

# -----------------------------------------------------------
# 1) Build the co-occurrence graph G (reuse edges already loaded in metric)
# -----------------------------------------------------------
edges_df = metric.edges_df

G = nx.Graph()
G.add_weighted_edges_from(
    edges_df[["ingredient_a", "ingredient_b", "weight"]].itertuples(index=False, name=None),
    weight="weight",
)

print("\nGraph rebuilt for Node2Vec training.")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# -----------------------------------------------------------
# 2) Iteration 9 hyperparameters (theory-consistent)
# -----------------------------------------------------------
ITER = 9

DIM = 128
WALK_LEN = 80        # <-- change vs iter8 (60 -> 80)
NUM_WALKS = 40       # keep (worked well)
P = 1
Q = 4                # keep DFS bias
WINDOW = 10          # keep (best so far)
SEED = 42

# Word2Vec training knobs (helps without breaking theory)
W2V_EPOCHS = 10       # default is often ~5; increase for stability/quality
SG = 1                # skip-gram (keep)
HS = 0                # negative sampling
NEGATIVE = 5          # typical default

# -----------------------------------------------------------
# 2b) Workers (speed)
# -----------------------------------------------------------
WALK_WORKERS = 8
W2V_WORKERS  = 8

# -----------------------------------------------------------
# 3) Train Node2Vec (UNWEIGHTED walks)
# -----------------------------------------------------------
node2vec_iter9 = Node2Vec(
    G,
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    # IMPORTANT: unweighted walks -> do NOT pass weight_key
    workers=WALK_WORKERS,
    seed=SEED,
)

model_iter9 = node2vec_iter9.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=W2V_WORKERS,
    epochs=W2V_EPOCHS,     # NEW
    sg=SG,                 # explicit
    hs=HS,                 # explicit
    negative=NEGATIVE,     # explicit
)

print(
    f"\nIteration {ITER} training complete "
    f"(UNWEIGHTED, p={P}, q={Q}, window={WINDOW}, walk_len={WALK_LEN}, num_walks={NUM_WALKS}, "
    f"epochs={W2V_EPOCHS}, walk_workers={WALK_WORKERS}, w2v_workers={W2V_WORKERS})."
)

# -----------------------------------------------------------
# 4) Save embeddings
# -----------------------------------------------------------
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter9 = OUT_DIR / (
    f"ingredient_node2vec_iter{ITER}_UNWEIGHTED_p{P}_q{Q}_w{WINDOW}_"
    f"wl{WALK_LEN}_r{NUM_WALKS}_e{W2V_EPOCHS}.txt"
)

model_iter9.wv.save_word2vec_format(str(emb_path_iter9))

print("Saved embeddings to:", emb_path_iter9)
print("Vectors:", len(model_iter9.wv), "| dim:", model_iter9.wv.vector_size)

# -----------------------------------------------------------
# 5) Run the metric on Iteration 9 (rank-aware)
# -----------------------------------------------------------
TOPK = 10
RANK_SCHEME = "log"  # keep consistent across iterations

score_iter9, per_q_iter9, meta_iter9 = metric.score(
    model_iter9.wv,
    topk=TOPK,
    rank_scheme=RANK_SCHEME,
    exclude_prefixes=("recipe::",),
    min_candidate_marg=0.0,
)

print("\n" + "=" * 70)
print(f"METRIC RESULT — Co-occurrence Node2Vec Iteration {ITER} (UNWEIGHTED) | rank={RANK_SCHEME}")
print(f"Eval set size: {meta_iter9['eval_set_size']}")
print(f"Scored queries: {meta_iter9['scored_queries']}")
print(f"Missing queries: {meta_iter9['missing_queries']}")
print(f"Empty candidate lists: {meta_iter9['empty_lists']}")
print(f"Top-k: {meta_iter9['topk']}")
print(f"ModelScore: {score_iter9:.6f}")
print("=" * 70)

if per_q_iter9:
    arr = np.array(per_q_iter9)
    print("\nPer-query Metric@k summary:")
    for pctl in [0, 10, 25, 50, 75, 90, 100]:
        print(f"  p{pctl:>3}: {np.percentile(arr, pctl):.6f}")

# -----------------------------------------------------------
# 6) Visual sanity check — Top-10 neighbours for a few probes
# -----------------------------------------------------------
def print_topk(kv, probes, topn=10):
    vocab = set(kv.key_to_index.keys())
    for p in probes:
        if p not in vocab:
            print(f"\nProbe missing: {p}")
            continue
        print("\n" + "=" * 60)
        print("Probe:", p)
        for i, (nbr, sim) in enumerate(kv.most_similar(p, topn=topn), 1):
            print(f"{i:02d}. {nbr:<30} cosine={sim:.3f}")

probes_to_check = [
    "real butter",
    "pure olive oil",
    "milk",
    "cream",
    "buttermilk",
    "flour",
    "cornstarch",
    "egg",
    "chicken",
    "vinegar",
]
print_topk(model_iter9.wv, probes_to_check, topn=10)



Graph rebuilt for Node2Vec training.
Nodes: 2698
Edges: 419913


Generating walks (CPU: 8): 100%|██████████| 5/5 [01:03<00:00, 12.74s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_


Iteration 9 training complete (UNWEIGHTED, p=1, q=4, window=10, walk_len=80, num_walks=40, epochs=10, walk_workers=8, w2v_workers=8).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter9_UNWEIGHTED_p1_q4_w10_wl80_r40_e10.txt
Vectors: 2698 | dim: 128

METRIC RESULT — Co-occurrence Node2Vec Iteration 9 (UNWEIGHTED) | rank=log
Eval set size: 300
Scored queries: 300
Missing queries: 0
Empty candidate lists: 0
Top-k: 10
ModelScore: 0.383931

Per-query Metric@k summary:
  p  0: 0.136867
  p 10: 0.215395
  p 25: 0.297684
  p 50: 0.384234
  p 75: 0.471259
  p 90: 0.530369
  p100: 0.684479

Probe: real butter
01. egg                            cosine=0.549
02. raspberry preserve             cosine=0.523
03. shortening                     cosine=0.510
04. soda                           cosine=0.506
05. whipping cream                 cosine=0.505
06. vanilla chip                   cosine=0.500
07. corn syrup                     cosine=0.499
08. margarine                   

In [9]:
# ===========================================================
# NEXT ITERATION (10) — Co-occurrence Node2Vec + metric scoring + sanity check
# CHANGE vs Iter 9: num_walks 40 → 80 (only change)
# Keep theory-aligned DFS bias: p=1, q=4
# ===========================================================

import networkx as nx
from node2vec import Node2Vec
from pathlib import Path
import numpy as np

# -----------------------------------------------------------
# 0) Ensure metric object exists & is built (tables only once)
# -----------------------------------------------------------
EDGES_PATH = Path("../data/processed/ingredient_cooccurrence_edges.csv")
EVAL_PATH  = Path("../data/processed/eval_ingredients_300_seed42.csv")

if "metric" not in globals():
    metric = CoocPPMIRankMetric(edges_path=EDGES_PATH, eval_path=EVAL_PATH, seed=42, n_eval=300)

if metric.eval_ingredients is None:
    metric.build(verbose=True)

# -----------------------------------------------------------
# 1) Build the co-occurrence graph G (reuse edges already loaded in metric)
# -----------------------------------------------------------
edges_df = metric.edges_df

G = nx.Graph()
G.add_weighted_edges_from(
    edges_df[["ingredient_a", "ingredient_b", "weight"]].itertuples(index=False, name=None),
    weight="weight",
)

print("\nGraph rebuilt for Node2Vec training.")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# -----------------------------------------------------------
# 2) Iteration 10 hyperparameters
# (same as Iter 9, ONLY num_walks increased)
# -----------------------------------------------------------
ITER = 10
DIM = 128
WALK_LEN = 80
NUM_WALKS = 80          # <-- ONLY change vs Iter 9 (40 -> 80)
P = 1
Q = 4
WINDOW = 10
EPOCHS = 10
SEED = 42

# -----------------------------------------------------------
# 2b) Workers (use your fast settings)
# -----------------------------------------------------------
WALK_WORKERS = 8
W2V_WORKERS  = 8

# -----------------------------------------------------------
# 3) Train Node2Vec (UNWEIGHTED walks)
# -----------------------------------------------------------
node2vec_iter10 = Node2Vec(
    G,
    dimensions=DIM,
    walk_length=WALK_LEN,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    # IMPORTANT: unweighted walks -> do NOT pass weight_key
    workers=WALK_WORKERS,
    seed=SEED,
)

model_iter10 = node2vec_iter10.fit(
    window=WINDOW,
    min_count=1,
    batch_words=4,
    workers=W2V_WORKERS,
    epochs=EPOCHS,
)

print(
    f"\nIteration {ITER} training complete "
    f"(UNWEIGHTED, p={P}, q={Q}, window={WINDOW}, walk_len={WALK_LEN}, "
    f"num_walks={NUM_WALKS}, epochs={EPOCHS}, walk_workers={WALK_WORKERS}, w2v_workers={W2V_WORKERS})."
)

# -----------------------------------------------------------
# 4) Save embeddings
# -----------------------------------------------------------
OUT_DIR = Path("../data/processed/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

emb_path_iter10 = OUT_DIR / (
    f"ingredient_node2vec_iter{ITER}_UNWEIGHTED_"
    f"p{P}_q{Q}_w{WINDOW}_wl{WALK_LEN}_r{NUM_WALKS}_e{EPOCHS}.txt"
)

model_iter10.wv.save_word2vec_format(str(emb_path_iter10))

print("Saved embeddings to:", emb_path_iter10)
print("Vectors:", len(model_iter10.wv), "| dim:", model_iter10.wv.vector_size)

# -----------------------------------------------------------
# 5) Run the metric on Iteration 10 (rank-aware)
# -----------------------------------------------------------
TOPK = 10
RANK_SCHEME = "log"

score_iter10, per_q_iter10, meta_iter10 = metric.score(
    model_iter10.wv,
    topk=TOPK,
    rank_scheme=RANK_SCHEME,
    exclude_prefixes=("recipe::",),
    min_candidate_marg=0.0,
)

print("\n" + "=" * 70)
print(f"METRIC RESULT — Co-occurrence Node2Vec Iteration {ITER} (UNWEIGHTED) | rank={RANK_SCHEME}")
print(f"Eval set size: {meta_iter10['eval_set_size']}")
print(f"Scored queries: {meta_iter10['scored_queries']}")
print(f"Missing queries: {meta_iter10['missing_queries']}")
print(f"Empty candidate lists: {meta_iter10['empty_lists']}")
print(f"Top-k: {meta_iter10['topk']}")
print(f"ModelScore: {score_iter10:.6f}")
print("=" * 70)

if per_q_iter10:
    arr = np.array(per_q_iter10)
    print("\nPer-query Metric@k summary:")
    for pctl in [0, 10, 25, 50, 75, 90, 100]:
        print(f"  p{pctl:>3}: {np.percentile(arr, pctl):.6f}")

# -----------------------------------------------------------
# 6) Visual sanity check — Top-10 neighbours for probes
# -----------------------------------------------------------
def print_topk(kv, probes, topn=10):
    vocab = set(kv.key_to_index.keys())
    for p in probes:
        if p not in vocab:
            print(f"\nProbe missing: {p}")
            continue
        print("\n" + "=" * 60)
        print("Probe:", p)
        for i, (nbr, sim) in enumerate(kv.most_similar(p, topn=topn), 1):
            print(f"{i:02d}. {nbr:<30} cosine={sim:.3f}")

probes_to_check = [
    "real butter",
    "pure olive oil",
    "milk",
    "cream",
    "buttermilk",
    "flour",
    "cornstarch",
    "egg",
    "chicken",
    "vinegar",
]
print_topk(model_iter10.wv, probes_to_check, topn=10)



Graph rebuilt for Node2Vec training.
Nodes: 2698
Edges: 419913


Generating walks (CPU: 3):  60%|██████    | 6/10 [03:21<02:33, 38.31s/it]/Users/adarpalaoglu/Documents/cs310-safeplate/venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Generating walks (CPU: 8): 100%|██████████| 10/10 [02:27<00:00, 14.74s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensi


Iteration 10 training complete (UNWEIGHTED, p=1, q=4, window=10, walk_len=80, num_walks=80, epochs=10, walk_workers=8, w2v_workers=8).
Saved embeddings to: ../data/processed/embeddings/ingredient_node2vec_iter10_UNWEIGHTED_p1_q4_w10_wl80_r80_e10.txt
Vectors: 2698 | dim: 128

METRIC RESULT — Co-occurrence Node2Vec Iteration 10 (UNWEIGHTED) | rank=log
Eval set size: 300
Scored queries: 300
Missing queries: 0
Empty candidate lists: 0
Top-k: 10
ModelScore: 0.388802

Per-query Metric@k summary:
  p  0: 0.153721
  p 10: 0.221324
  p 25: 0.307211
  p 50: 0.384975
  p 75: 0.470201
  p 90: 0.528093
  p100: 0.675979

Probe: real butter
01. flour                          cosine=0.625
02. shortening                     cosine=0.618
03. unsifted flour                 cosine=0.607
04. soda                           cosine=0.596
05. buttermilk                     cosine=0.596
06. powder                         cosine=0.588
07. sifted flour                   cosine=0.577
08. nut                      